In [ ]:
!pip install "bayesian-optimization==3.3.0"
!pip install "hyperopt==0.3.0"
!pip install "ConfigSpace==1.2.2"
!pip install "smac==2.4.0"

import time
import numpy as np
import matplotlib.pyplot as plt
from bayes_opt import BayesianOptimization
from bayes_opt.acquisition import ExpectedImprovement
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from ConfigSpace import ConfigurationSpace, Float

from smac import HyperparameterOptimizationFacade, Scenario
from smac.callback.callback import Callback

from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests
import itertools
import re
import pandas as pd
from IPython.display import FileLink, display
from IPython.display import display
import importlib.metadata as importlib_metadata
from functools import partial
from pprint import pprint

from sklearn.gaussian_process.kernels import Matern


In [ ]:
# Here are many different families of patients
baseline_family = {
    'name': 'quick_rise_slow_fall',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

symmetric_family = {
    'name': 'symmetric',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'symmetric',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

slow_rise_quick_fall_family = {
    'name': 'slow_rise_quick_fall',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'slow_rise_quick_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

three_optima_family = {
    'name': '3optima',
    'num_optima': 3,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

broad_peaks_family = {
    'name': 'broad_optima',
    'num_optima': 2,
    'spread_scale': 2.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

sharp_peaks_family = {
    'name': 'sharp_optima',
    'num_optima': 2,
    'spread_scale': 0.4,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

heteroscedastic_noise_family = {
    'name': 'heteroscedastic_noise',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'heteroscedastic',
    'unique_parameter_interaction': 'none'
}

heavy_tailed_noise_family = {
    'name': 'heavy_tailed_noise',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'heavy_tailed',
    'unique_parameter_interaction': 'none'
}

unique_parameter_interaction_family = {
    'name': 'unique_parameter_interaction',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'pulsewidth-amplitude'
}


def asymmetric_hill_peak(
    value,
    optimum,
    left_scale,
    right_scale,
    hill_n=4.0,
):
    if left_scale <= 0 or right_scale <= 0 or hill_n <= 0:
        raise ValueError("Scales and hill_n must be positive.")

    value = np.asarray(value, dtype=float)
    scale = np.where(value < optimum, left_scale, right_scale)
    distance = np.abs(value - optimum) / scale

    return 1.0 / (1.0 + distance**hill_n)


def generate_patient_profile(patient_seed: int, family: dict):
    rng = np.random.default_rng(patient_seed)

    base_optima = [(10.0 + rng.uniform(-2.0, 2.0), 500.0 + rng.uniform(-25.0, 25.0)),
                   (25.0 + rng.uniform(-2.0, 2.0), 100.0 + rng.uniform(-25.0, 25.0))] #may need to change this later on.
    num_optima = family['num_optima']
    optima = []
    for numOfOptima in range(num_optima):
        if numOfOptima < len(base_optima):
            opt_f, opt_w = base_optima[numOfOptima]
        else:
            opt_f = rng.uniform(5.0, 50.0)
            opt_w = rng.uniform(150.0, 450.0)
        optima.append((opt_f, opt_w))

    spread_scale = family['spread_scale']

    if family['function_shape'] == 'quick_rise_slow_fall':
        return {
            'patient_seed': patient_seed,
            'surface_family': family['name'],
            'ideal_amp_pup': rng.normal(4.4, 0.15),
            'ideal_amp_hrv': rng.normal(4.0, 0.15),
            # Asymmetric spreads: faster rise toward threshold, slower decline afterwards
            'amp_rise_pup': rng.uniform(0.25, 0.35) * spread_scale,
            'amp_fall_pup': rng.uniform(0.55, 0.75) * spread_scale,
            'amp_rise_hrv': rng.uniform(0.20, 0.30) * spread_scale,
            'amp_fall_hrv': rng.uniform(0.45, 0.65) * spread_scale,

            'freq_rise': rng.uniform(1.0, 2.0) * spread_scale,
            'freq_fall': rng.uniform(5.0, 6.0) * spread_scale,
            'pulse_width_rise': rng.uniform(20.0, 50.0) * spread_scale,
            'pulse_width_fall': rng.uniform(125.0, 175.0) * spread_scale,

            'optima': optima,
        }
    elif family['function_shape'] == 'symmetric':
        return {
            'patient_seed': patient_seed,
            'surface_family': family['name'],
            'ideal_amp_pup': rng.normal(4.4, 0.15),
            'ideal_amp_hrv': rng.normal(4.0, 0.15),
            # Symmetric spreads: same rise and fall
            'amp_rise_pup': 0.25,
            'amp_fall_pup': 0.25,
            'amp_rise_hrv': 0.25,
            'amp_fall_hrv': 0.25,
            'freq_rise': 1.5,
            'freq_fall': 1.5,
            'pulse_width_rise': 75.0,
            'pulse_width_fall': 75.0,

            'optima': optima,
        }
    elif family['function_shape'] == 'slow_rise_quick_fall':
        return {
            'patient_seed': patient_seed,
            'surface_family': family['name'],
            'ideal_amp_pup': rng.normal(4.4, 0.15),
            'ideal_amp_hrv': rng.normal(4.0, 0.15),
            # Asymmetric spreads: slow rise toward threshold, faster fall afterwards
            'amp_rise_pup': rng.uniform(0.55, 0.75) * spread_scale,
            'amp_fall_pup': rng.uniform(0.25, 0.35) * spread_scale,
            'amp_rise_hrv': rng.uniform(0.45, 0.65) * spread_scale,
            'amp_fall_hrv': rng.uniform(0.20, 0.30) * spread_scale,

            'freq_rise': rng.uniform(5.0, 6.0) * spread_scale,
            'freq_fall': rng.uniform(1.0, 2.0) * spread_scale,
            'pulse_width_rise': rng.uniform(125.0, 175.0) * spread_scale,
            'pulse_width_fall': rng.uniform(20.0, 50.0) * spread_scale,

            'optima': optima,
        }


# Convergence is based on the parameters each method selects from noisy observations.
# The oracle curve is reported separately and is never returned to an optimizer.
INCUMBENT_PARAMETER_COLUMNS = {
    "amplitude": "Selected Incumbent Amplitude (mA)",
    "frequency": "Selected Incumbent Frequency (Hz)",
    "pulse_width": "Selected Incumbent Pulse Width (us)",
}
SELECTED_INCUMBENT_REGRET_COLUMN = "Selected Incumbent Normalized Regret (%)"
ORACLE_BEST_EVALUATED_REGRET_COLUMN = "Oracle Best-Evaluated Normalized Regret (%)"
INCUMBENT_POLICY_COLUMN = "Incumbent Selection Policy"
OBSERVED_INCUMBENT_POLICY = "highest observed score"
SMAC_INCUMBENT_POLICY = "SMAC intensifier"
SMAC_METHOD = "SMAC Optimization"


def _physical_parameters(params):
    return {name: float(params[name]) for name in INCUMBENT_PARAMETER_COLUMNS}


def _normalize_parameters(params, bounds):
    return {
        name: (float(params[name]) - lower) / (upper - lower)
        for name, (lower, upper) in bounds.items()
    }


def _denormalize_parameters(params, bounds):
    return {
        name: lower + float(params[name]) * (upper - lower)
        for name, (lower, upper) in bounds.items()
    }


def _update_observed_incumbent(best_score, best_params, score, params):
    # Strict comparison retains the first point on a tie, as these methods do.
    if score > best_score:
        return score, _physical_parameters(params)
    return best_score, best_params


def _set_incumbent_parameters(row, params):
    if params is None:
        raise RuntimeError("No incumbent was selected from the observed data.")
    for name, column in INCUMBENT_PARAMETER_COLUMNS.items():
        row[column] = float(params[name])


def _normalized_regret_percent(best_possible_score, score):
    if best_possible_score <= 0:
        return np.nan
    return 100 * (best_possible_score - score) / best_possible_score


class SMACIncumbentRecorder(Callback):
    """Record SMAC's own incumbent after its intensifier has processed a trial."""

    def __init__(self):
        super().__init__()
        self.records = []

    def on_tell_end(self, smbo, info, value):
        incumbent = smbo.intensifier.get_incumbent()
        if incumbent is None:
            raise RuntimeError("SMAC did not select an incumbent after a completed trial.")
        # Only observed costs and parameter values are handled by this callback.
        self.records.append({
            "trial_params": _physical_parameters(info.config),
            "incumbent_params": _physical_parameters(incumbent),
            "cost": float(value.cost),
        })


def _attach_smac_incumbents(rows, records, finished_trials):
    """Align each completed SMAC trial with its actual post-tell incumbent."""
    smac_rows = [row for row in rows if row["Method"] == SMAC_METHOD]
    if len(smac_rows) != len(records) or finished_trials != len(smac_rows):
        raise RuntimeError(
            "Cannot reconstruct the complete SMAC incumbent trajectory: trial, "
            "callback, and evaluation counts differ. Execute the optimizer "
            "configuration cell before running the comparison."
        )
    for query, (row, record) in enumerate(zip(smac_rows, records), start=1):
        row_params = (row["Amplitude"], row["Frequency"], row["Pulse Width"])
        trial_params = tuple(record["trial_params"][name] for name in INCUMBENT_PARAMETER_COLUMNS)
        if (
            row["Query"] != query
            or not np.allclose(row_params, trial_params, rtol=0, atol=1e-12)
            or not np.isclose(row["Observed Score"], -record["cost"], rtol=1e-12, atol=1e-12)
        ):
            raise RuntimeError("SMAC trial records do not match the recorded evaluations.")
        _set_incumbent_parameters(row, record["incumbent_params"])


def _finalize_convergence_metrics(rows, best_possible_score):
    """Calculate selected and oracle regrets from diagnostic scores, after selection."""
    evaluated_scores = {}
    for row in rows:
        evaluated_params = (row["Amplitude"], row["Frequency"], row["Pulse Width"])
        evaluated_scores[(row["Method"], evaluated_params)] = row["Noise-Free Score"]
        selected_params = tuple(row[column] for column in INCUMBENT_PARAMETER_COLUMNS.values())
        selected_key = (row["Method"], selected_params)
        if selected_key not in evaluated_scores:
            raise RuntimeError("A selected incumbent was not evaluated by that query.")
        incumbent_score = evaluated_scores[selected_key]
        row["Current Incumbent Noise-Free Score"] = incumbent_score
        selected_regret = _normalized_regret_percent(best_possible_score, incumbent_score)
        row[SELECTED_INCUMBENT_REGRET_COLUMN] = selected_regret
        # Backward-compatible name now explicitly aliases the selected-incumbent metric.
        row["Normalized Regret So Far (%)"] = selected_regret
        row[ORACLE_BEST_EVALUATED_REGRET_COLUMN] = _normalized_regret_percent(
            best_possible_score, row["Best Noise-Free Score So Far"]
        )


def _validate_final_convergence(results_df, convergence_df):
    """Require the final trajectory point to describe the reported recommendation."""
    final_rows = convergence_df.sort_values("Query").groupby("Method", sort=False).tail(1)
    final_rows = final_rows.set_index("Method", verify_integrity=True)
    for _, result in results_df.iterrows():
        method = result["Method"]
        if method not in final_rows.index:
            raise RuntimeError(f"No final convergence row for {method}.")
        final = final_rows.loc[method]
        for convergence_column, result_column in (
            ("Selected Incumbent Amplitude (mA)", "Selected Amplitude (mA)"),
            ("Selected Incumbent Frequency (Hz)", "Selected Frequency (Hz)"),
            ("Selected Incumbent Pulse Width (us)", "Selected Pulse Width (us)"),
            ("Current Incumbent Noise-Free Score", "Selected Score (Noise-Free)"),
            (SELECTED_INCUMBENT_REGRET_COLUMN, "Normalized Regret (%)"),
            (SELECTED_INCUMBENT_REGRET_COLUMN, SELECTED_INCUMBENT_REGRET_COLUMN),
            (ORACLE_BEST_EVALUATED_REGRET_COLUMN, ORACLE_BEST_EVALUATED_REGRET_COLUMN),
        ):
            if not np.isclose(final[convergence_column], result[result_column],
                              rtol=1e-12, atol=1e-12, equal_nan=True):
                raise RuntimeError(f"Final convergence does not match {result_column} for {method}.")


SIMULATED_GAUSSIAN_NOISE_STD = 2.0
SIMULATED_HEAVY_TAILED_DF = 3
SIMULATED_HEAVY_TAILED_SCALE = 2.0


def run_optimization_comparison(num_simulation_trials: int, amount_of_patients: int, patient_and_machine_startSeed : int, patient_type: dict):

    search_boundaries = {
        'amplitude': (0.5, 5.0),       # mA
        'frequency': (1.0, 50.0),      # Hz
        'pulse_width': (50.0, 600.0),  # microseconds
    }

    bo_normalized_bounds = {
        name: (0.0, 1.0) for name in search_boundaries
    }

    all_patients_results = [] 
    all_convergence_results = []

    for i in range(amount_of_patients):

        patient_profile = generate_patient_profile(patient_and_machine_startSeed, patient_type)
        ideal_amp_pup = patient_profile['ideal_amp_pup']
        amp_rise_pup = patient_profile['amp_rise_pup']
        amp_fall_pup = patient_profile['amp_fall_pup']
        ideal_amp_hrv = patient_profile['ideal_amp_hrv']
        amp_rise_hrv = patient_profile['amp_rise_hrv']
        amp_fall_hrv = patient_profile['amp_fall_hrv']
        freq_rise = patient_profile['freq_rise']
        freq_fall = patient_profile['freq_fall']
        pulse_width_rise = patient_profile['pulse_width_rise']
        pulse_width_fall = patient_profile['pulse_width_fall']
        optima = patient_profile['optima']

        def realistic_patient_curve(amplitude, frequency, pulse_width, AddNoise: bool, patient_and_machine_startSeed: int):

            amp_comp_pup = asymmetric_hill_peak(
                amplitude,
                ideal_amp_pup,
                amp_rise_pup,
                amp_fall_pup,
            )
            amp_comp_hrv = asymmetric_hill_peak(
                amplitude,
                ideal_amp_hrv,
                amp_rise_hrv,
                amp_fall_hrv,
            )

            pup_score = 100 * amp_comp_pup
            hrv_score = 100 * amp_comp_hrv
            w_pup, w_hrv = 0.5, 0.5
            amplitude_score = w_pup * pup_score + w_hrv * hrv_score

            def get_interdependence_score(f, w, Amplitude_Scale_Factor = 1.0):
                scores = []
                for (opt_f, opt_w) in optima:
                    freq_comp = asymmetric_hill_peak(
                        f,
                        opt_f,
                        freq_rise,
                        freq_fall,
                    )
                    
                    width_comp = asymmetric_hill_peak(
                        w,
                        opt_w,
                        pulse_width_rise,
                        pulse_width_fall,
                    )
                    dist = ((freq_comp) * (width_comp)) * Amplitude_Scale_Factor
                    scores.append(dist)

                return np.max(np.asarray(scores), axis=0)

            def get_interdependence_score_for_special_interaction(f, w, a):
                scores = []
                for (opt_f, opt_w) in optima:
                    if opt_f >= 25:
                        freq_comp = asymmetric_hill_peak(
                            f,
                            opt_f - a,
                            freq_rise,
                            freq_fall,
                        )
                    else:
                        freq_comp = asymmetric_hill_peak(
                            f,
                            opt_f + a,
                            freq_rise,
                            freq_fall,
                        )

                    if opt_w >= 350: 
                        width_comp = asymmetric_hill_peak(
                            w,
                            opt_w - a * 20,
                            pulse_width_rise,
                            pulse_width_fall,
                        )
                    else:
                        width_comp = asymmetric_hill_peak(
                            w,
                            opt_w + a * 20,
                            pulse_width_rise,
                            pulse_width_fall,
                        )
                    dist = ((freq_comp) * (width_comp)) 
                    scores.append(dist)
            
                return np.max(np.asarray(scores), axis=0)
            
            if patient_type.get('unique_parameter_interaction') == 'pulsewidth-amplitude':
                inter_score = get_interdependence_score_for_special_interaction(frequency, pulse_width, amplitude)
            else:
                inter_score = get_interdependence_score(frequency, pulse_width)

            total_score = inter_score * amplitude_score

            if not AddNoise:
                return total_score

            rng = np.random.default_rng(patient_and_machine_startSeed)
            noise_type = patient_type.get('noise', 'normal')
            if noise_type == 'normal':
                noise = rng.normal(0, SIMULATED_GAUSSIAN_NOISE_STD)
            elif noise_type == 'heteroscedastic':
                    # Noise scales with signal magnitude
                noise = rng.normal(0, amplitude)
            elif noise_type == 'heavy_tailed':
                    # Student's t-distribution
                noise = rng.standard_t(df=SIMULATED_HEAVY_TAILED_DF) * SIMULATED_HEAVY_TAILED_SCALE
            else:
                noise = 0.0

            total_score += noise

            return total_score

        #Find the optimal score and parameters
        if patient_type.get('unique_parameter_interaction') != 'pulsewidth-amplitude':
            amplitude_grid = np.linspace(search_boundaries['amplitude'][0], search_boundaries['amplitude'][1], 10000)
            max_possible_score = -np.inf
            for (opt_f, opt_w) in optima:
                # Vectorized evaluation over the amplitude grid at the peak frequency & pulse width
                curve_scores = realistic_patient_curve(amplitude_grid, opt_f, opt_w, False, 0)
                best_idx = np.argmax(curve_scores)
                peak_score = curve_scores[best_idx]

                if peak_score > max_possible_score:
                    max_possible_score = peak_score
        else: 
            amplitude_grid = np.linspace(
                search_boundaries['amplitude'][0],
                search_boundaries['amplitude'][1],
                10000,
            )
            max_possible_score = -np.inf

            for opt_f, opt_w in optima:
                if opt_f >= 25:
                    shifted_frequency = opt_f - amplitude_grid
                else:
                    shifted_frequency = opt_f + amplitude_grid

                if opt_w >= 350:
                    shifted_pulse_width = opt_w - 20 * amplitude_grid
                else:
                    shifted_pulse_width = opt_w + 20 * amplitude_grid

                curve_scores = realistic_patient_curve(
                    amplitude_grid,
                    shifted_frequency,
                    shifted_pulse_width,
                    False,
                    0,
                )
                peak_score = np.max(curve_scores)

                if peak_score > max_possible_score:
                    max_possible_score = peak_score
        

        method_names = (
            "Bayesian Optimization",
            "TPE Optimization",
            "SMAC Optimization",
            "Random Search",
        )

        def make_evaluation_seed(profile_seed, query_index):
            sequence = np.random.SeedSequence(
                [int(profile_seed), int(query_index)]
            )
            return int(sequence.generate_state(1, dtype=np.uint32)[0])

        profile_seed = patient_and_machine_startSeed
        convergence_rows = []

        query_counts = {
            method: 0
            for method in method_names
        }

        best_observed = {
            method: -np.inf
            for method in method_names
        }

        best_noise_free = {
            method: -np.inf
            for method in method_names
        }

        observed_incumbent_params = {
            method: None
            for method in method_names
        }

        def evaluate_and_record(method, params, evaluation_seed):
            query_index = query_counts[method] + 1
            query_counts[method] = query_index

            observed_score = realistic_patient_curve(
                params["amplitude"],
                params["frequency"],
                params["pulse_width"],
                True,
                evaluation_seed,
            )

            noise_free_score = realistic_patient_curve(
                params["amplitude"],
                params["frequency"],
                params["pulse_width"],
                False,
                evaluation_seed,
            )

            best_observed[method], observed_incumbent_params[method] = (
                _update_observed_incumbent(
                    best_observed[method],
                    observed_incumbent_params[method],
                    observed_score,
                    params,
                )
            )

            best_noise_free[method] = max(
                best_noise_free[method],
                noise_free_score,
            )

            row = {
                "Profile ID": (
                    f"{patient_profile['surface_family']}::{profile_seed}"
                ),
                "Surface Family": patient_profile["surface_family"],
                "Profile Seed": profile_seed,
                "Iteration Budget": num_simulation_trials,
                "Method": method,
                "Query": query_index,
                "Evaluation Seed": evaluation_seed,
                "Amplitude": params["amplitude"],
                "Frequency": params["frequency"],
                "Pulse Width": params["pulse_width"],
                "Observed Score": observed_score,
                "Noise-Free Score": noise_free_score,
                "Cumulative Best Observed Score": best_observed[method],
                "Best Noise-Free Score So Far": best_noise_free[method],
                INCUMBENT_POLICY_COLUMN: (
                    SMAC_INCUMBENT_POLICY if method == SMAC_METHOD
                    else OBSERVED_INCUMBENT_POLICY
                ),
            }
            if method != SMAC_METHOD:
                _set_incumbent_parameters(row, observed_incumbent_params[method])
            convergence_rows.append(row)

            # The optimizer receives the noisy observation only.
            return observed_score

        def objective_bo(amplitude, frequency, pulse_width):
            query_index = (
                query_counts["Bayesian Optimization"] + 1
            )
            physical_params = _denormalize_parameters(
                {
                    "amplitude": amplitude,
                    "frequency": frequency,
                    "pulse_width": pulse_width,
                },
                search_boundaries,
            )
            evaluation_seed = make_evaluation_seed(
                profile_seed,
                query_index,
            )

            return evaluate_and_record(
                "Bayesian Optimization",
                physical_params,
                evaluation_seed,
            )

        optimizer_multi = BayesianOptimization(
            f=objective_bo,
            pbounds=bo_normalized_bounds,
            gp_noise_type=patient_type.get('noise', 'normal'),
            gp_physical_bounds=search_boundaries,
            acquisition_function=ExpectedImprovement(xi=0.01),
            random_state=patient_and_machine_startSeed,
            verbose=0
        )
        print(f"\n--- Starting Bayesian Optimization ({num_simulation_trials} iterations) ---")
        bo_start_time = time.perf_counter()
        optimizer_multi.maximize(
            init_points=BOGP_DEFAULTS["init_points"],
            n_iter=num_simulation_trials - BOGP_DEFAULTS["init_points"],
        )

        bo_selected_params = _denormalize_parameters(
            optimizer_multi.max["params"], search_boundaries
        )

        bo_elapsed_seconds = time.perf_counter() - bo_start_time
        print(f"Bayesian Optimization completed in {bo_elapsed_seconds:.3f} seconds.")

        # --- TPE Optimization ---
        tpe_start_time = time.perf_counter()
        def objective_tpe(params):
            query_index = query_counts["TPE Optimization"] + 1
            evaluation_seed = make_evaluation_seed(
                profile_seed,
                query_index,
            )

            score = evaluate_and_record(
                "TPE Optimization",
                {
                    "amplitude": params["amplitude"],
                    "frequency": params["frequency"],
                    "pulse_width": params["pulse_width"],
                },
                evaluation_seed,
            )

            return {
                "loss": -score,
                "status": STATUS_OK,
            }
        space = {
            'amplitude': hp.uniform('amplitude', search_boundaries['amplitude'][0], search_boundaries['amplitude'][1]),
            'frequency': hp.uniform('frequency', search_boundaries['frequency'][0], search_boundaries['frequency'][1]),
            'pulse_width': hp.uniform('pulse_width', search_boundaries['pulse_width'][0], search_boundaries['pulse_width'][1])
        }
        trials = Trials()
        print(f"\n--- Starting TPE Optimization ({num_simulation_trials} iterations) ---")
        fmin(fn=objective_tpe, space=space, algo=tpe.suggest, max_evals=num_simulation_trials, trials=trials, rstate=np.random.default_rng(patient_and_machine_startSeed))
        tpe_elapsed_seconds = time.perf_counter() - tpe_start_time
        print(f"TPE completed in {tpe_elapsed_seconds:.3f} seconds.")

        # --- SMAC Optimization ---
        smac_start_time = time.perf_counter()
        configspace = ConfigurationSpace(seed=patient_and_machine_startSeed)
        configspace.add([
            Float('amplitude', bounds=search_boundaries['amplitude']),
            Float('frequency', bounds=search_boundaries['frequency']),
            Float('pulse_width', bounds=search_boundaries['pulse_width']),
        ])

        def objective_smac(config, seed=None):
            query_index = query_counts["SMAC Optimization"] + 1
            evaluation_seed = make_evaluation_seed(
                profile_seed,
                query_index,
            )
            score = evaluate_and_record(
                "SMAC Optimization",
                {
                    "amplitude": float(config["amplitude"]),
                    "frequency": float(config["frequency"]),
                    "pulse_width": float(config["pulse_width"]),
                },
                evaluation_seed,
            )

            return -score

        print(f"\n--- Starting SMAC Optimization ({num_simulation_trials} iterations) ---")
        scenario = Scenario(
            configspace,
            deterministic=SMAC_DEFAULTS["scenario"]["deterministic"],
            objectives=SMAC_DEFAULTS["scenario"]["objectives"],
            use_default_config=SMAC_DEFAULTS["scenario"]["use_default_config"],
            n_workers=SMAC_DEFAULTS["scenario"]["n_workers"],
            n_trials=num_simulation_trials,
            seed=patient_and_machine_startSeed,
        )
        smac_incumbents = SMACIncumbentRecorder()
        smac = HyperparameterOptimizationFacade(
            scenario, objective_smac, callbacks=[smac_incumbents]
        )
        incumbent = smac.optimize()
        smac_score = -smac.runhistory.get_cost(incumbent)

        smac_elapsed_seconds = time.perf_counter() - smac_start_time
        _attach_smac_incumbents(
            convergence_rows, smac_incumbents.records, smac.runhistory.finished
        )
        print(f"SMAC completed in {smac_elapsed_seconds:.3f} seconds.")

        # --- Random Search ---
        rs_start_time = time.perf_counter()
        random_search_rng = np.random.default_rng(patient_and_machine_startSeed)
        best_random_score = -np.inf
        best_random_params = {}
        print(f"\n--- Starting Random Search Optimization ({num_simulation_trials} iterations) ---")
        for _ in range(num_simulation_trials):
            a, f, w = [
                random_search_rng.uniform(
                    search_boundaries[key][0],
                    search_boundaries[key][1],
                )
                for key in [
                    "amplitude",
                    "frequency",
                    "pulse_width",
                ]
            ]

            query_index = query_counts["Random Search"] + 1
            evaluation_seed = make_evaluation_seed(
                profile_seed,
                query_index,
            )

            current_score = evaluate_and_record(
                "Random Search",
                {
                    "amplitude": a,
                    "frequency": f,
                    "pulse_width": w,
                },
                evaluation_seed,
            )

            if current_score > best_random_score:
                best_random_score = current_score
                best_random_params = {
                    "amplitude": a,
                    "frequency": f,
                    "pulse_width": w,
                }

        rs_elapsed_seconds = time.perf_counter() - rs_start_time
        print(f"Random Search completed in {rs_elapsed_seconds:.3f} seconds.")

        incomplete_query_counts = {
            method: query_counts[method]
            for method in method_names
            if query_counts[method] != num_simulation_trials
        }
        if incomplete_query_counts:
            raise RuntimeError(
                f"Optimizer objective evaluation counts differ from the requested "
                f"budget of {num_simulation_trials}: {incomplete_query_counts}."
            )
        if smac.runhistory.finished != num_simulation_trials:
            raise RuntimeError(
                f"SMAC completed {smac.runhistory.finished} runhistory evaluations; "
                f"expected {num_simulation_trials}."
            )

        _finalize_convergence_metrics(convergence_rows, max_possible_score)

        selected_scores_noise_free = [
            realistic_patient_curve(
                bo_selected_params["amplitude"],
                bo_selected_params["frequency"],
                bo_selected_params["pulse_width"],
                False,
                patient_and_machine_startSeed,
            ),
            realistic_patient_curve(
                trials.argmin['amplitude'],
                trials.argmin['frequency'],
                trials.argmin['pulse_width'],
                False,
                patient_and_machine_startSeed,
            ),
            realistic_patient_curve(
                float(incumbent['amplitude']),
                float(incumbent['frequency']),
                float(incumbent['pulse_width']),
                False,
                patient_and_machine_startSeed,
            ),
            realistic_patient_curve(
                best_random_params['amplitude'],
                best_random_params['frequency'],
                best_random_params['pulse_width'],
                False,
                patient_and_machine_startSeed,
            ),
        ]

        simple_regrets = [
            max_possible_score - score
            for score in selected_scores_noise_free
        ]

        normalized_regrets = [
            regret / max_possible_score
            if max_possible_score > 0
            else np.nan
            for regret in simple_regrets
        ]

        comparison_data = {
            'Method': ['Bayesian Optimization', 'TPE Optimization', 'SMAC Optimization', 'Random Search'],
            'Surface Family': [patient_profile['surface_family']] * 4,
            'Seed': [patient_and_machine_startSeed] * 4,
            'Best Possible Score': [max_possible_score] * 4,
            'Best Computed Score': [optimizer_multi.max['target'], -trials.best_trial['result']['loss'], smac_score, best_random_score],
            'Selected Amplitude (mA)': [bo_selected_params["amplitude"], trials.argmin['amplitude'], float(incumbent['amplitude']), best_random_params['amplitude']],
            'Selected Frequency (Hz)': [bo_selected_params["frequency"], trials.argmin['frequency'], float(incumbent['frequency']), best_random_params['frequency']],
            'Selected Pulse Width (us)': [bo_selected_params["pulse_width"], trials.argmin['pulse_width'], float(incumbent['pulse_width']), best_random_params['pulse_width']],

            'Selected Score (Noise-Free)': selected_scores_noise_free,
            'Simple Regret': simple_regrets,
            #Gives percentage regret just in case
            'Normalized Regret (%)': [
                                100 * regret
                                for regret in normalized_regrets
                            ],
            'Sec To Compute': [bo_elapsed_seconds, tpe_elapsed_seconds, smac_elapsed_seconds, rs_elapsed_seconds]
        }

        # Explicit names make the primary recommendation and oracle diagnostic distinct.
        comparison_data[SELECTED_INCUMBENT_REGRET_COLUMN] = comparison_data["Normalized Regret (%)"].copy()
        comparison_data[ORACLE_BEST_EVALUATED_REGRET_COLUMN] = [
            _normalized_regret_percent(max_possible_score, best_noise_free[method])
            for method in comparison_data["Method"]
        ]
        patient_results = pd.DataFrame(comparison_data)
        patient_convergence = pd.DataFrame(convergence_rows)
        _validate_final_convergence(patient_results, patient_convergence)
        all_patients_results.append(patient_results)
        all_convergence_results.append(patient_convergence)

        patient_and_machine_startSeed += 1

    return all_patients_results, all_convergence_results

In [ ]:
# This section contains a standalone clinical-data-anchored response function.
# Edit these bounds for your clinical protocol, or pass bounds to the builder.
CLINICAL_MAP_FREQUENCY_BOUNDS_HZ = (1.0, 50.0)
CLINICAL_MAP_PULSE_WIDTH_BOUNDS_US = (50.0, 600.0)
# Provisional only: validate this normalized radius against clinical evidence.
CLINICAL_MAP_INFLUENCE_RADIUS = 0.25

# Enter 1-10 measured rows as (frequency_hz, pulse_width_us, response_value).
# Replace this commented template with real clinical measurements.
CLINICAL_RESPONSE_ANCHORS = [
    # (frequency_hz, pulse_width_us, response_value),
]


def _validate_clinical_map_bounds(bounds, name):
    try:
        values = np.asarray(bounds, dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{name} must contain two finite numeric bounds.") from error
    if values.shape != (2,) or not np.isfinite(values).all() or values[0] >= values[1]:
        raise ValueError(f"{name} must be a finite (lower, upper) pair with lower < upper.")
    return float(values[0]), float(values[1])


def build_clinical_response_function(
    anchor_points,
    frequency_bounds_hz=None,
    pulse_width_bounds_us=None,
    influence_radius=None,
):
    """Build a local 2D response interpolator from 1-10 measured anchors.

    Each anchor is ``(frequency_hz, pulse_width_us, response_value)``. The
    response is returned in the same units as the supplied measurements;
    zero is treated as no effect. Frequency and pulse width are normalized
    by their bounds before computing Euclidean distances.

    Nearby anchors are blended with inverse-square distance weights, then a
    compact smooth taper makes the response reach exactly zero at the
    influence radius from the nearest anchor. This is a local, phenomenological
    assumption, not a mechanistic or causal biological model. The default
    radius (0.25 normalized units) is provisional and should be sensitivity-
    tested against clinical knowledge. Queries outside the supplied bounds
    are rejected rather than extrapolated.

    The returned callable accepts scalars or broadcastable NumPy arrays. To
    create a surface, evaluate it on ``numpy.meshgrid`` arrays.
    """
    try:
        anchors = np.asarray(anchor_points, dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError("anchor_points must be numeric (frequency, pulse width, response) rows.") from error

    if anchors.ndim == 1:
        anchors = anchors.reshape(1, -1)
    if anchors.ndim != 2 or anchors.shape[1] != 3:
        raise ValueError("anchor_points must have shape (n, 3): frequency_hz, pulse_width_us, response_value.")
    if not 1 <= anchors.shape[0] <= 10:
        raise ValueError("Provide between 1 and 10 clinical anchor points.")
    if not np.isfinite(anchors).all():
        raise ValueError("anchor_points must contain only finite numeric values.")

    if frequency_bounds_hz is None:
        frequency_bounds_hz = CLINICAL_MAP_FREQUENCY_BOUNDS_HZ
    if pulse_width_bounds_us is None:
        pulse_width_bounds_us = CLINICAL_MAP_PULSE_WIDTH_BOUNDS_US
    frequency_lower, frequency_upper = _validate_clinical_map_bounds(
        frequency_bounds_hz, "frequency_bounds_hz"
    )
    pulse_width_lower, pulse_width_upper = _validate_clinical_map_bounds(
        pulse_width_bounds_us, "pulse_width_bounds_us"
    )

    if influence_radius is None:
        influence_radius = CLINICAL_MAP_INFLUENCE_RADIUS
    try:
        influence_radius = float(influence_radius)
    except (TypeError, ValueError) as error:
        raise ValueError("influence_radius must be a finite positive normalized distance.") from error
    if (
        not np.isfinite(influence_radius)
        or influence_radius <= 0
        or influence_radius > np.sqrt(2.0)
    ):
        raise ValueError("influence_radius must be greater than zero and at most sqrt(2).")

    if (
        np.any(anchors[:, 0] < frequency_lower)
        or np.any(anchors[:, 0] > frequency_upper)
        or np.any(anchors[:, 1] < pulse_width_lower)
        or np.any(anchors[:, 1] > pulse_width_upper)
    ):
        raise ValueError("Every anchor must lie within the supplied frequency and pulse-width bounds.")

    anchor_coordinates = np.column_stack((
        (anchors[:, 0] - frequency_lower) / (frequency_upper - frequency_lower),
        (anchors[:, 1] - pulse_width_lower) / (pulse_width_upper - pulse_width_lower),
    ))
    if np.unique(anchor_coordinates, axis=0).shape[0] != len(anchor_coordinates):
        raise ValueError("Anchor frequency/pulse-width coordinates must be unique.")
    anchor_responses = anchors[:, 2].copy()

    def smooth_compact_taper(distance_ratio):
        """Quintic smoothstep that is zero at and beyond the support radius."""
        ratio = np.clip(distance_ratio, 0.0, 1.0)
        return 1.0 - 10.0 * ratio**3 + 15.0 * ratio**4 - 6.0 * ratio**5

    def response_function(frequency_hz, pulse_width_us):
        try:
            frequencies, pulse_widths = np.broadcast_arrays(
                np.asarray(frequency_hz, dtype=float),
                np.asarray(pulse_width_us, dtype=float),
            )
        except (TypeError, ValueError) as error:
            raise ValueError("Frequency and pulse-width queries must be numeric and broadcastable.") from error
        if not np.isfinite(frequencies).all() or not np.isfinite(pulse_widths).all():
            raise ValueError("Frequency and pulse-width queries must be finite.")
        if (
            np.any(frequencies < frequency_lower)
            or np.any(frequencies > frequency_upper)
            or np.any(pulse_widths < pulse_width_lower)
            or np.any(pulse_widths > pulse_width_upper)
        ):
            raise ValueError("Queries must lie within the supplied frequency and pulse-width bounds.")

        query_coordinates = np.column_stack((
            ((frequencies - frequency_lower) / (frequency_upper - frequency_lower)).ravel(),
            ((pulse_widths - pulse_width_lower) / (pulse_width_upper - pulse_width_lower)).ravel(),
        ))
        interpolated = np.zeros(len(query_coordinates), dtype=float)
        exact_anchor_tolerance = 1e-12

        for query_index, query in enumerate(query_coordinates):
            distances = np.linalg.norm(anchor_coordinates - query, axis=1)
            nearest_index = int(np.argmin(distances))
            nearest_distance = distances[nearest_index]
            if nearest_distance <= exact_anchor_tolerance:
                interpolated[query_index] = anchor_responses[nearest_index]
                continue
            if nearest_distance >= influence_radius:
                continue

            supported = distances < influence_radius
            supported_distances = distances[supported]
            weights = smooth_compact_taper(
                supported_distances / influence_radius
            ) / supported_distances**2
            local_response = np.average(anchor_responses[supported], weights=weights)
            falloff = smooth_compact_taper(nearest_distance / influence_radius)
            interpolated[query_index] = falloff * local_response

        response_values = interpolated.reshape(frequencies.shape)
        if response_values.ndim == 0:
            return float(response_values.item())
        return response_values

    return response_function


# Populated automatically after you enter measured anchors above.
CLINICAL_RESPONSE_FUNCTION = (
    build_clinical_response_function(CLINICAL_RESPONSE_ANCHORS)
    if CLINICAL_RESPONSE_ANCHORS
    else None
)

In [ ]:
OPTIMIZER_STARTUP_POINTS = 4

BOGP_DEFAULTS = {
    "acquisition_function": "ExpectedImprovement",
    "xi": 0.01,
    "exploration_decay": None,
    "exploration_decay_delay": None,
    "acquisition_random_state": None,
    "gp_input_bounds": "[0.0, 1.0] independently for all three inputs",
    "gp_length_scale": 1.0,
    "gp_kernel": "Matern(length_scale=1.0, nu=2.5) on normalized inputs",
    "gp_alpha_original": 1e-6,
    "gp_alpha_policy": "per-observation variance / std(y)^2 + numerical jitter",
    "gp_numerical_jitter": 1e-6,
    "gp_normalize_y": True,
    "gp_noise_variances": {
        "normal": "4.0 score^2 (Gaussian standard deviation 2.0)",
        "heteroscedastic": "physical amplitude_i^2 score^2 per observation",
        "heavy_tailed": "12.0 score^2 (Student-t df=3, scale=2; moment matched)",
    },
    "gp_heavy_tailed_limitation": "Gaussian likelihood does not model Student-t tails",
    "gp_n_restarts_optimizer": 5,
    "gp_optimizer": "fmin_l_bfgs_b",
    "init_points": OPTIMIZER_STARTUP_POINTS,
    "n_iter_rule": "num_simulation_trials - init_points",
    "random_state": "patient_and_machine_startSeed",
    "verbose": 0,
    "allow_duplicate_points": True,
}


def _gp_observation_noise_variances(noise_type, physical_amplitudes):
    amplitudes = np.asarray(physical_amplitudes, dtype=float)
    if noise_type == "normal":
        return np.full_like(amplitudes, SIMULATED_GAUSSIAN_NOISE_STD ** 2, dtype=float)
    if noise_type == "heteroscedastic":
        return np.square(amplitudes)
    if noise_type == "heavy_tailed":
        variance = (
            SIMULATED_HEAVY_TAILED_SCALE ** 2
            * SIMULATED_HEAVY_TAILED_DF
            / (SIMULATED_HEAVY_TAILED_DF - 2)
        )
        return np.full_like(amplitudes, variance, dtype=float)
    return np.zeros_like(amplitudes, dtype=float)


def _normalized_gp_alpha(noise_variances, targets, normalize_y):
    """Express observation variance in the target units used by normalize_y."""
    noise_variances = np.asarray(noise_variances, dtype=float)
    if not normalize_y:
        return noise_variances
    target_scale = float(np.std(np.asarray(targets, dtype=float)))
    if target_scale < 10 * np.finfo(float).eps:
        target_scale = 1.0
    return noise_variances / target_scale ** 2


TPE_DEFAULTS = {
    "prior_weight": 1.0,
    "n_startup_jobs": OPTIMIZER_STARTUP_POINTS,
    "n_EI_candidates": 24,
    "gamma": 0.25,
    "suggest_verbose": False,
    "fmin_verbose": False,
    "show_progressbar": False,
    "max_queue_len": 1,
    "return_argmin": True,
    "allow_trials_fmin": True,
    "rstate": "np.random.default_rng(patient_and_machine_startSeed)",
}

SMAC_DEFAULTS = {
    "scenario": {
        "deterministic": False,
        "objectives": "cost",
        "use_default_config": False,
        "n_workers": 1,
        "n_trials": "num_simulation_trials",
        "seed": "patient_and_machine_startSeed",
    },
    "model": {
        "type": "RandomForest",
        "n_trees": 10,
        "ratio_features": 1.0,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_depth": 1048576,
        "bootstrapping": True,
    },
    "acquisition_function": {
        "type": "Expected Improvement",
        "xi": 0.0,
    },
    "initial_design": {
        "type": "SobolInitialDesign",
        "n_configs": OPTIMIZER_STARTUP_POINTS,
        "max_ratio": 1.0,
        "additional_configs": None,
    },
    "random_design": {
        "type": "ProbabilityRandomDesign",
        "probability": 0.2,
    },
    "intensifier": {
        "type": "Intensifier",
        "max_config_calls": 3,
        "max_incumbents": 10,
    },
    "config_selector": {
        "retrain_after": 8,
        "retries": 16,
    },
}

RANDOM_SEARCH_DEFAULTS = {
    "distribution": "independent Uniform(low, high) per parameter",
    "bounds": {
        "amplitude": (0.5, 5.0),
        "frequency": (1.0, 50.0),
        "pulse_width": (50.0, 600.0),
    },
    "sampling": "one independent draw for each parameter at every query",
    "replacement": True,
    "generator": "np.random.default_rng",
    "seed": "patient_and_machine_startSeed",
    "query_budget": "num_simulation_trials",
}

OPTIMIZER_PACKAGE_VERSIONS = {
    package_name: importlib_metadata.version(package_name)
    for package_name in (
        "bayesian-optimization",
        "hyperopt",
        "ConfigSpace",
        "smac",
    )
}


if "_original_bayesian_optimization" not in globals():
    _original_bayesian_optimization = BayesianOptimization


class ConfiguredBayesianOptimization(_original_bayesian_optimization):
    def __init__(self, *args, **kwargs):
        self._gp_noise_type = kwargs.pop("gp_noise_type", "normal")
        self._gp_physical_bounds = kwargs.pop("gp_physical_bounds", None)
        if self._gp_noise_type == "heteroscedastic" and (
            self._gp_physical_bounds is None
            or "amplitude" not in self._gp_physical_bounds
        ):
            raise ValueError("Heteroscedastic GP noise requires physical amplitude bounds.")
        kwargs.setdefault(
            "allow_duplicate_points",
            BOGP_DEFAULTS["allow_duplicate_points"],
        )
        super().__init__(*args, **kwargs)
        self.set_gp_params(
            kernel=Matern(length_scale=BOGP_DEFAULTS["gp_length_scale"], nu=2.5),
            alpha=BOGP_DEFAULTS["gp_numerical_jitter"],
            normalize_y=BOGP_DEFAULTS["gp_normalize_y"],
            n_restarts_optimizer=BOGP_DEFAULTS["gp_n_restarts_optimizer"],
            optimizer=BOGP_DEFAULTS["gp_optimizer"],
        )

    def _update_observation_noise_alpha(self):
        targets = np.asarray(self._space.target, dtype=float)
        if targets.size == 0:
            return
        if self._gp_noise_type == "heteroscedastic":
            amplitude_index = self._space.keys.index("amplitude")
            normalized_amplitudes = self._space.params[:, amplitude_index]
            lower, upper = self._gp_physical_bounds["amplitude"]
            physical_amplitudes = lower + normalized_amplitudes * (upper - lower)
        else:
            physical_amplitudes = np.zeros_like(targets)
        noise_variances = _gp_observation_noise_variances(
            self._gp_noise_type, physical_amplitudes
        )
        alpha = _normalized_gp_alpha(
            noise_variances, targets, BOGP_DEFAULTS["gp_normalize_y"]
        )
        self._gp.set_params(
            alpha=alpha + BOGP_DEFAULTS["gp_numerical_jitter"]
        )

    def suggest(self):
        self._update_observation_noise_alpha()
        return super().suggest()


BayesianOptimization = ConfiguredBayesianOptimization


if "_original_tpe_suggest" not in globals():
    _original_tpe_suggest = tpe.suggest

tpe.suggest = partial(
    _original_tpe_suggest,
    prior_weight=TPE_DEFAULTS["prior_weight"],
    n_startup_jobs=TPE_DEFAULTS["n_startup_jobs"],
    n_EI_candidates=TPE_DEFAULTS["n_EI_candidates"],
    gamma=TPE_DEFAULTS["gamma"],
    verbose=TPE_DEFAULTS["suggest_verbose"],
)


if "_original_hyperopt_fmin" not in globals():
    _original_hyperopt_fmin = fmin


def configured_hyperopt_fmin(*args, **kwargs):
    kwargs.setdefault("verbose", TPE_DEFAULTS["fmin_verbose"])
    kwargs.setdefault("show_progressbar", TPE_DEFAULTS["show_progressbar"])
    kwargs.setdefault("max_queue_len", TPE_DEFAULTS["max_queue_len"])
    kwargs.setdefault("return_argmin", TPE_DEFAULTS["return_argmin"])
    kwargs.setdefault("allow_trials_fmin", TPE_DEFAULTS["allow_trials_fmin"])
    return _original_hyperopt_fmin(*args, **kwargs)


fmin = configured_hyperopt_fmin


if "_original_smac_facade" not in globals():
    _original_smac_facade = HyperparameterOptimizationFacade


class ConfiguredSMACFacade(_original_smac_facade):
    def __init__(self, scenario, target_function=None, **kwargs):
        kwargs.setdefault("overwrite", True)
        kwargs.setdefault(
            "model",
            _original_smac_facade.get_model(
                scenario,
                n_trees=SMAC_DEFAULTS["model"]["n_trees"],
                ratio_features=SMAC_DEFAULTS["model"]["ratio_features"],
                min_samples_split=SMAC_DEFAULTS["model"]["min_samples_split"],
                min_samples_leaf=SMAC_DEFAULTS["model"]["min_samples_leaf"],
                max_depth=SMAC_DEFAULTS["model"]["max_depth"],
                bootstrapping=SMAC_DEFAULTS["model"]["bootstrapping"],
            ),
        )
        kwargs.setdefault(
            "acquisition_function",
            _original_smac_facade.get_acquisition_function(
                scenario,
                xi=SMAC_DEFAULTS["acquisition_function"]["xi"],
            ),
        )
        kwargs.setdefault(
            "initial_design",
            _original_smac_facade.get_initial_design(
                scenario,
                n_configs=SMAC_DEFAULTS["initial_design"]["n_configs"],
                max_ratio=SMAC_DEFAULTS["initial_design"]["max_ratio"],
                additional_configs=SMAC_DEFAULTS["initial_design"][
                    "additional_configs"
                ],
            ),
        )
        kwargs.setdefault(
            "random_design",
            _original_smac_facade.get_random_design(
                scenario,
                probability=SMAC_DEFAULTS["random_design"]["probability"],
            ),
        )
        kwargs.setdefault(
            "intensifier",
            _original_smac_facade.get_intensifier(
                scenario,
                max_config_calls=SMAC_DEFAULTS["intensifier"]["max_config_calls"],
                max_incumbents=SMAC_DEFAULTS["intensifier"]["max_incumbents"],
            ),
        )
        kwargs.setdefault(
            "config_selector",
            _original_smac_facade.get_config_selector(
                scenario,
                retrain_after=SMAC_DEFAULTS["config_selector"]["retrain_after"],
                retries=SMAC_DEFAULTS["config_selector"]["retries"],
            ),
        )
        super().__init__(scenario, target_function, **kwargs)


HyperparameterOptimizationFacade = ConfiguredSMACFacade


print("Explicit optimizer configuration applied.")
print("Installed optimizer package versions:")
pprint(OPTIMIZER_PACKAGE_VERSIONS)
print("\nBOGP defaults:")
pprint(BOGP_DEFAULTS)
print("\nTPE defaults:")
pprint(TPE_DEFAULTS)
print("\nSMAC defaults:")
pprint(SMAC_DEFAULTS)
print("\nRandom Search defaults:")
pprint(RANDOM_SEARCH_DEFAULTS)

### GP observation-noise specification

`GaussianProcessRegressor.alpha` is a variance added to the covariance diagonal. With `normalize_y=True`, scikit-learn first divides targets by their current training-set standard deviation, so alpha must be expressed in normalized-target variance units. Scikit-learn accepts a per-observation alpha array; this notebook recomputes it before each GP fit as `alpha_i = sigma_i^2 / std(y_train)^2 + 1e-6`, where `1e-6` is numerical jitter. See the [GaussianProcessRegressor API](https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.GaussianProcessRegressor.html).

- **Original:** `alpha=1e-6`, `normalize_y=True`. This treated observation noise as approximately `1e-6` normalized score variance (standard deviation `0.001`), rather than reflecting the simulator's noise.
- **Corrected baseline Gaussian model:** the baseline uses `N(0, 2.0^2)` score noise, so its known observation variance is `4.0` score-squared units; each fit uses `4.0 / std(y_train)^2 + 1e-6`.
- **Heteroscedastic model:** the simulator uses `N(0, amplitude_i)` noise. The GP receives a separate variance `amplitude_i^2 / std(y_train)^2 + 1e-6` for each training observation, with amplitude converted back to physical units.
- **Heavy-tailed model:** the simulator uses `Student-t(df=3) * 2.0`, whose finite variance is `12.0`; that variance is moment-matched in alpha. The GP still has a Gaussian likelihood, so this does not model Student-t tails or protect predictions from outliers.

The GP allows duplicate parameter inputs because this is a noisy objective: evaluating a repeated configuration obtains a fresh seeded observation, and skipping duplicate proposals would silently reduce the requested query budget. This setting is scoped to Bayesian Optimization; TPE, SMAC, and random search are unchanged.

These noise variances are derived from the simulator's definitions, not selected by optimizer performance. The previous `1e-6` is retained only as numerical jitter in the corrected model.

### Shared startup budget

The GP-based Bayesian Optimization method, TPE, and SMAC each receive four initial evaluations before model-guided proposals. The Bayesian optimizer uses `init_points=4` and then fits its Gaussian Process to those observations; TPE uses `n_startup_jobs=4`; SMAC uses a fixed four-configuration Sobol initial design. SMAC's initial-design ratio cap is set to `1.0` so it does not reduce the fixed count for smaller supported budgets. The random-search baseline remains random for all queries because it has no model-guided phase.

### SMAC budget and incumbent policy

SMAC's `Scenario` explicitly uses the configured single-worker, stochastic, single-cost settings, with `n_trials=num_simulation_trials` and the profile seed. Its target returns the negative noisy score for minimization. The target derives its noise seed from the profile seed and that method's query number, rather than SMAC's per-run `seed`; this gives all methods the same seeded noise draw at a given query, while the Scenario seed controls SMAC's optimizer randomness. The notebook verifies that each optimizer records exactly the requested number of objective calls and that SMAC's finished-run count agrees, including repeated configurations.

`SMACIncumbentRecorder` captures the intensifier's incumbent after each completed trial, rather than replacing it with the best single observed score. SMAC's `RunHistory.get_cost(incumbent)` is the cached empirical mean over its unique instance/seed/budget run keys, so repeated evaluations contribute to the incumbent cost. The SMAC policy is labeled separately from the other methods' highest-observed-score policy, and the final recommendation is checked against the last convergence incumbent.

In [ ]:
def export_family_results(
    family,
    family_test,
    family_convergence,
    iteration_budget=40,
    output_dir=".",
    file_prefix=None,
    n_permutations=20000,
    n_bootstrap=10000,
    random_seed=10001,
    display_links=True,
):
    """Analyze one family's optimizer results and export its CSV outputs."""
    if isinstance(family, dict):
        family_name = family.get("name")
    else:
        family_name = str(family)

    if not isinstance(family_name, str) or not family_name.strip():
        raise ValueError("family must be a non-empty name or a dict with a 'name'.")
    if not isinstance(family_test, (list, tuple)) or not family_test:
        raise ValueError("family_test must be a non-empty list of per-profile DataFrames.")
    if not isinstance(family_convergence, (list, tuple)):
        raise ValueError("family_convergence must be a list of per-profile DataFrames.")
    if len(family_test) != len(family_convergence):
        raise ValueError("family_test and family_convergence must have the same length.")
    if not all(isinstance(frame, pd.DataFrame) for frame in family_test):
        raise TypeError("Every item in family_test must be a pandas DataFrame.")
    if not all(isinstance(frame, pd.DataFrame) for frame in family_convergence):
        raise TypeError("Every item in family_convergence must be a pandas DataFrame.")

    required_columns = {
        "Surface Family",
        "Seed",
        "Method",
        "Best Possible Score",
        "Selected Score (Noise-Free)",
    }
    missing_columns = set().union(
        *(required_columns.difference(frame.columns) for frame in family_test)
    )
    if missing_columns:
        raise ValueError(f"Missing family result columns: {sorted(missing_columns)}")

    family_results_df = pd.concat(
        [
            result.assign(
                **{
                    "Profile Index": profile_index,
                    "Profile ID": result["Surface Family"].astype(str)
                    + "::"
                    + result["Seed"].astype(str),
                    "Iteration Budget": iteration_budget,
                }
            )
            for profile_index, result in enumerate(family_test, start=1)
        ],
        ignore_index=True,
    )

    observed_families = set(family_results_df["Surface Family"].astype(str).unique())
    if observed_families != {family_name}:
        raise ValueError(
            f"family name {family_name!r} does not match result families {sorted(observed_families)}."
        )

    family_convergence_df = pd.concat(
        [
            convergence.assign(**{"Profile Index": profile_index})
            for profile_index, convergence in enumerate(family_convergence, start=1)
        ],
        ignore_index=True,
    )

    analysis_df = family_results_df.copy()
    analysis_df["Normalized Regret"] = (
        analysis_df["Best Possible Score"]
        - analysis_df["Selected Score (Noise-Free)"]
    ) / analysis_df["Best Possible Score"]

    method_order = [
        "Bayesian Optimization",
        "TPE Optimization",
        "SMAC Optimization",
        "Random Search",
    ]
    methods = [
        method
        for method in method_order
        if method in analysis_df["Method"].unique()
    ]
    if len(methods) != len(method_order):
        raise ValueError(f"Expected all four methods; found {methods}.")

    duplicate_keys = analysis_df.duplicated(
        subset=["Profile ID", "Method"],
        keep=False,
    )
    if duplicate_keys.any():
        raise ValueError(
            "Each family profile-method combination must have exactly one result."
        )

    family_wide = analysis_df.pivot(
        index="Profile ID",
        columns="Method",
        values="Normalized Regret",
    ).reindex(columns=methods).dropna()

    if len(family_wide) < 2:
        raise ValueError("At least two complete paired profiles are required.")

    def omnibus_permutation_test(wide, seed):
        values = wide.to_numpy(dtype=float)
        observed_statistic = float(np.ptp(values.mean(axis=0)))
        rng = np.random.default_rng(seed)
        permuted_statistics = np.empty(n_permutations)

        for permutation_index in range(n_permutations):
            permuted_values = np.array(
                [rng.permutation(profile_values) for profile_values in values]
            )
            permuted_statistics[permutation_index] = np.ptp(
                permuted_values.mean(axis=0)
            )

        p_value = (
            1
            + np.count_nonzero(permuted_statistics >= observed_statistic)
        ) / (n_permutations + 1)
        return observed_statistic, p_value

    def paired_sign_flip_test(first, second, seed):
        differences = np.asarray(first, dtype=float) - np.asarray(second, dtype=float)
        observed_difference = float(np.mean(differences))
        rng = np.random.default_rng(seed)
        signs = rng.choice(
            np.array([-1.0, 1.0]),
            size=(n_permutations, len(differences)),
        )
        permuted_means = np.mean(signs * differences, axis=1)
        p_value = (
            1
            + np.count_nonzero(
                np.abs(permuted_means) >= abs(observed_difference)
            )
        ) / (n_permutations + 1)
        return observed_difference, p_value

    def paired_bootstrap_ci(differences, seed):
        differences = np.asarray(differences, dtype=float)
        rng = np.random.default_rng(seed)
        indices = rng.integers(
            0,
            len(differences),
            size=(n_bootstrap, len(differences)),
        )
        bootstrap_means = differences[indices].mean(axis=1)
        return tuple(np.quantile(bootstrap_means, [0.025, 0.975]))

    def holm_adjust(p_values):
        p_values = np.asarray(p_values, dtype=float)
        order = np.argsort(p_values)
        sorted_p_values = p_values[order]
        adjusted_sorted = np.maximum.accumulate(
            (len(sorted_p_values) - np.arange(len(sorted_p_values)))
            * sorted_p_values
        )
        adjusted = np.empty_like(adjusted_sorted)
        adjusted[order] = np.minimum(adjusted_sorted, 1.0)
        return adjusted

    friedman_statistic, friedman_p_value = stats.friedmanchisquare(
        *[family_wide[method].to_numpy() for method in methods]
    )
    omnibus_statistic, omnibus_p_value = omnibus_permutation_test(
        family_wide,
        seed=random_seed + iteration_budget,
    )
    kendalls_w = friedman_statistic / (
        len(family_wide) * (len(methods) - 1)
    )

    omnibus_df = pd.DataFrame(
        [
            {
                "Surface Family": family_name,
                "Iteration Budget": iteration_budget,
                "N Paired Profiles": len(family_wide),
                "Endpoint": "Noise-free normalized regret",
                "Friedman Statistic": friedman_statistic,
                "Friedman P Value": friedman_p_value,
                "Kendalls W": kendalls_w,
                "Permutation Range Statistic": omnibus_statistic,
                "Paired Permutation P Value": omnibus_p_value,
            }
        ]
    )

    pairwise_rows = []
    for pair_index, (method_a, method_b) in enumerate(
        itertools.combinations(methods, 2)
    ):
        differences = (
            family_wide[method_a].to_numpy()
            - family_wide[method_b].to_numpy()
        )
        mean_difference, p_value = paired_sign_flip_test(
            family_wide[method_a],
            family_wide[method_b],
            seed=random_seed + 100000 + pair_index,
        )
        ci_low, ci_high = paired_bootstrap_ci(
            differences,
            seed=random_seed + 200000 + pair_index,
        )
        difference_sd = np.std(differences, ddof=1)
        pairwise_rows.append(
            {
                "Surface Family": family_name,
                "Iteration Budget": iteration_budget,
                "Method A": method_a,
                "Method B": method_b,
                "N Paired Profiles": len(differences),
                "Mean Regret Difference (A - B)": mean_difference,
                "95% Bootstrap CI Low": ci_low,
                "95% Bootstrap CI High": ci_high,
                "Paired Effect Size dz": (
                    mean_difference / difference_sd
                    if difference_sd > 0
                    else np.nan
                ),
                "P Method A Has Lower Regret": np.mean(differences < 0),
                "P Value": p_value,
            }
        )

    pairwise_df = pd.DataFrame(pairwise_rows)
    pairwise_df["Holm Adjusted P Value"] = holm_adjust(
        pairwise_df["P Value"]
    )

    summary_df = (
        analysis_df.groupby("Method")["Normalized Regret"]
        .agg(
            N_Profiles="count",
            Mean_Regret="mean",
            SD_Regret="std",
            Median_Regret="median",
            Q25_Regret=lambda values: values.quantile(0.25),
            Q75_Regret=lambda values: values.quantile(0.75),
        )
        .reindex(methods)
        .reset_index()
    )

    safe_prefix = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(file_prefix or family_name).strip(),
    ).strip("_")
    if not safe_prefix:
        raise ValueError("file_prefix must contain at least one usable character.")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    family_title = safe_prefix.replace("_", " ").title()
    download_files = {
        f"{family_title} profile results": (
            output_dir / f"{safe_prefix}_family_results.csv",
            family_results_df,
        ),
        f"{family_title} convergence results": (
            output_dir / f"{safe_prefix}_family_convergence.csv",
            family_convergence_df,
        ),
        f"{family_title} descriptive statistics": (
            output_dir / f"{safe_prefix}_summary_statistics.csv",
            summary_df,
        ),
        f"{family_title} omnibus tests": (
            output_dir / f"{safe_prefix}_omnibus_tests.csv",
            omnibus_df,
        ),
        f"{family_title} paired post-hoc tests": (
            output_dir / f"{safe_prefix}_pairwise_tests.csv",
            pairwise_df,
        ),
    }

    for path, frame in download_files.values():
        frame.to_csv(path, index=False)

    print(
        f"Exported {family_name!r}: {len(family_results_df):,} profile-method rows, "
        f"{len(family_convergence_df):,} query rows, and "
        f"{len(download_files)} CSV files."
    )
    if display_links:
        for label, (path, _) in download_files.items():
            display(FileLink(str(path), result_html_prefix=f"{label}: "))

    return {
        "family_name": family_name,
        "results_df": family_results_df,
        "convergence_df": family_convergence_df,
        "analysis_df": analysis_df,
        "wide": family_wide,
        "summary_df": summary_df,
        "omnibus_df": omnibus_df,
        "pairwise_df": pairwise_df,
        "files": {label: str(path) for label, (path, _) in download_files.items()},
    }


def run_and_export_family(
    family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    iteration_budget=None,
    output_dir=".",
    file_prefix=None,
    **export_kwargs,
):
    """Run one family through all optimizers, analyze it, and export its CSVs."""
    family_test, family_convergence = run_optimization_comparison(
        num_simulation_trials=num_simulation_trials,
        amount_of_patients=amount_of_patients,
        patient_and_machine_startSeed=patient_and_machine_startSeed,
        patient_type=family,
    )
    outputs = export_family_results(
        family=family,
        family_test=family_test,
        family_convergence=family_convergence,
        iteration_budget=(
            num_simulation_trials
            if iteration_budget is None
            else iteration_budget
        ),
        output_dir=output_dir,
        file_prefix=file_prefix,
        **export_kwargs,
    )
    outputs["family_test"] = family_test
    outputs["family_convergence"] = family_convergence
    return outputs

In [ ]:
METHOD_ORDER = [
    "Bayesian Optimization",
    "TPE Optimization",
    "SMAC Optimization",
    "Random Search",
]

# Queries each optimizer spends on its initial design before model-based search begins.
# Random Search never switches to a model, so it has no boundary.
INITIAL_DESIGN_QUERIES = {
    "Bayesian Optimization": 4,
    "TPE Optimization": 20,
    "SMAC Optimization": 10,
    "Random Search": None,
}

INCUMBENT_SCORE_COLUMN = "Current Incumbent Noise-Free Score"
INCUMBENT_REGRET_COLUMN = SELECTED_INCUMBENT_REGRET_COLUMN
ORACLE_REGRET_COLUMN = ORACLE_BEST_EVALUATED_REGRET_COLUMN


def _build_common_random_number_summary(common_random_number_df):
    summary_columns = [
        "Iteration Budget",
        "N Tested Groups",
        "N Passed Groups",
        "N Failed Groups",
        "Overall Status",
    ]
    if common_random_number_df.empty:
        return pd.DataFrame(columns=summary_columns)

    summary_df = (
        common_random_number_df.groupby("Iteration Budget")[
            "Uses Common Random Number"
        ]
        .agg(
            **{
                "N Tested Groups": "size",
                "N Passed Groups": "sum",
                "N Failed Groups": lambda values: (~values).sum(),
            }
        )
        .reset_index()
    )
    summary_df["N Passed Groups"] = summary_df["N Passed Groups"].astype(int)
    summary_df["N Failed Groups"] = summary_df["N Failed Groups"].astype(int)
    summary_df["Overall Status"] = np.where(
        summary_df["N Failed Groups"] == 0,
        "PASS",
        "FAIL",
    )
    return summary_df


def _add_incumbent_regret(convergence_df, best_possible_by_profile):
    """Validate the recommendation trajectory; never substitute an oracle curve."""
    if convergence_df.empty:
        return None

    required = {
        "Profile ID", "Method", INCUMBENT_SCORE_COLUMN,
        INCUMBENT_REGRET_COLUMN, INCUMBENT_POLICY_COLUMN,
        *INCUMBENT_PARAMETER_COLUMNS.values(),
    }
    missing = required.difference(convergence_df.columns)
    if missing:
        raise ValueError(
            "Selected-incumbent convergence data are incomplete or from an older "
            "notebook. Earlier logs did not track SMAC's actual per-query "
            "incumbent; regenerate convergence data before analyzing trajectories. "
            f"Missing columns: {sorted(missing)}. Final results can still be "
            "analyzed without convergence data."
        )

    expected_policy = convergence_df["Method"].map({
        "Bayesian Optimization": OBSERVED_INCUMBENT_POLICY,
        "TPE Optimization": OBSERVED_INCUMBENT_POLICY,
        "SMAC Optimization": SMAC_INCUMBENT_POLICY,
        "Random Search": OBSERVED_INCUMBENT_POLICY,
    })
    if expected_policy.isna().any() or not expected_policy.eq(
        convergence_df[INCUMBENT_POLICY_COLUMN]
    ).all():
        raise ValueError("Convergence rows have an unknown or incorrect incumbent selection policy.")

    best_possible = pd.to_numeric(
        convergence_df["Profile ID"].map(best_possible_by_profile), errors="coerce"
    )
    incumbent_score = pd.to_numeric(convergence_df[INCUMBENT_SCORE_COLUMN], errors="coerce")
    recorded_regret = pd.to_numeric(convergence_df[INCUMBENT_REGRET_COLUMN], errors="coerce")
    usable = (
        np.isfinite(best_possible) & (best_possible > 0)
        & np.isfinite(incumbent_score) & np.isfinite(recorded_regret)
    )
    if not usable.all():
        raise ValueError("Selected-incumbent convergence contains missing or invalid values.")
    expected_regret = 100 * (best_possible - incumbent_score) / best_possible
    if not np.allclose(recorded_regret, expected_regret, rtol=1e-10, atol=1e-10):
        raise ValueError("Recorded selected-incumbent regret does not match the incumbent score.")
    if "Normalized Regret So Far (%)" in convergence_df and not np.allclose(
        pd.to_numeric(convergence_df["Normalized Regret So Far (%)"], errors="coerce"),
        recorded_regret, rtol=1e-10, atol=1e-10,
    ):
        raise ValueError("The legacy-named regret alias does not match selected-incumbent regret.")

    convergence_df["Best Possible Score"] = best_possible
    return INCUMBENT_REGRET_COLUMN


def _build_convergence_summary(trajectory_summary_df):
    summary_columns = [
        "Surface Family",
        "Method",
        "Iteration Budget",
        "N Profiles",
        "N Queries",
        "First Query",
        "Final Query",
        "First Median Normalized Regret (%)",
        "Final Median Normalized Regret (%)",
        "Final Q25 (%)",
        "Final Q75 (%)",
        "Median Improvement (%)",
    ]
    if trajectory_summary_df.empty:
        return pd.DataFrame(columns=summary_columns)

    group_columns = ["Surface Family", "Method", "Iteration Budget"]
    summary_rows = []
    for group_values, group in trajectory_summary_df.groupby(group_columns, sort=False):
        ordered_group = group.sort_values("Query")
        first_row = ordered_group.iloc[0]
        final_row = ordered_group.iloc[-1]
        summary_rows.append(
            {
                **dict(zip(group_columns, group_values)),
                "N Profiles": (
                    int(final_row["N Profiles"])
                    if "N Profiles" in ordered_group
                    else np.nan
                ),
                "N Queries": len(ordered_group),
                "First Query": first_row["Query"],
                "Final Query": final_row["Query"],
                "First Median Normalized Regret (%)": first_row["Median"],
                "Final Median Normalized Regret (%)": final_row["Median"],
                "Final Q25 (%)": final_row["Q25"],
                "Final Q75 (%)": final_row["Q75"],
                "Median Improvement (%)": first_row["Median"] - final_row["Median"],
            }
        )

    return pd.DataFrame(summary_rows, columns=summary_columns).sort_values(
        group_columns
    ).reset_index(drop=True)


def analyze_family_results(
    family_data,
    family_convergence=None,
    *,
    family_name=None,
    iteration_budget=40,
    score_column=None,
    regret_tolerance=0.05,
    n_permutations=20000,
    n_bootstrap=10000,
    random_seed=10001,
    display_results=True,
    plot_results=True,
):
    """Analyze one family using normalized noise-free regret as the primary endpoint."""
    if n_permutations < 1 or n_bootstrap < 1:
        raise ValueError("n_permutations and n_bootstrap must both be positive.")

    if isinstance(family_data, dict):
        result_data = family_data.get("results_df", family_data.get("family_test"))
        if result_data is None:
            raise ValueError("family_data must contain 'results_df' or 'family_test'.")
        if family_convergence is None:
            family_convergence = family_data.get(
                "convergence_df", family_data.get("family_convergence")
            )
        if family_name is None:
            family_name = family_data.get("family_name")
    else:
        result_data = family_data

    if isinstance(result_data, pd.DataFrame):
        analysis_df = result_data.copy()
    elif isinstance(result_data, (list, tuple)) and result_data:
        result_frames = []
        for profile_index, result_frame in enumerate(result_data, start=1):
            if not isinstance(result_frame, pd.DataFrame):
                raise TypeError("Each family result must be a pandas DataFrame.")
            result_frame = result_frame.copy()
            if "Profile Index" not in result_frame:
                result_frame["Profile Index"] = profile_index
            if "Iteration Budget" not in result_frame:
                result_frame["Iteration Budget"] = iteration_budget
            result_frames.append(result_frame)
        analysis_df = pd.concat(result_frames, ignore_index=True)
    else:
        raise TypeError(
            "family_data must be a results DataFrame, a non-empty list of DataFrames, "
            "or the dictionary returned by run_and_export_family."
        )

    required_columns = {"Surface Family", "Seed", "Method", "Best Possible Score"}
    missing_columns = required_columns.difference(analysis_df.columns)
    if missing_columns:
        raise ValueError(f"Missing required result columns: {sorted(missing_columns)}")

    score_candidates = ["Selected Score (Noise-Free)", "Independent Reevaluation Score"]
    if score_column is None:
        score_column = next(
            (
                candidate
                for candidate in score_candidates
                if candidate in analysis_df and not analysis_df[candidate].isna().all()
            ),
            None,
        )
    if score_column is None or score_column not in analysis_df:
        raise ValueError(
            "A selected noise-free or independent reevaluation score is required. "
            "The noisy maximum in 'Best Computed Score' is not an acceptable primary endpoint."
        )
    if score_column == "Best Computed Score":
        raise ValueError("Best Computed Score is a noisy maximum and cannot be primary.")

    analysis_df["Surface Family"] = analysis_df["Surface Family"].astype(str)
    observed_families = set(analysis_df["Surface Family"].unique())
    if family_name is None:
        if len(observed_families) != 1:
            raise ValueError("family_name is required when results contain more than one family.")
        family_name = next(iter(observed_families))
    family_name = str(family_name)
    if observed_families != {family_name}:
        raise ValueError(
            f"family_name {family_name!r} does not match result families "
            f"{sorted(observed_families)}."
        )

    if "Iteration Budget" not in analysis_df:
        analysis_df["Iteration Budget"] = iteration_budget
    analysis_df["Iteration Budget"] = pd.to_numeric(
        analysis_df["Iteration Budget"], errors="coerce"
    )
    if analysis_df["Iteration Budget"].isna().any():
        raise ValueError("Iteration Budget must be numeric for every result row.")
    analysis_df["Iteration Budget"] = analysis_df["Iteration Budget"].astype(int)

    if "Profile ID" not in analysis_df:
        analysis_df["Profile ID"] = (
            analysis_df["Surface Family"] + "::" + analysis_df["Seed"].astype(str)
        )
    else:
        analysis_df["Profile ID"] = analysis_df["Profile ID"].astype(str)

    numeric_columns = ["Best Possible Score", score_column]
    if "Sec To Compute" in analysis_df:
        numeric_columns.append("Sec To Compute")
    for column in numeric_columns:
        analysis_df[column] = pd.to_numeric(analysis_df[column], errors="coerce")
    if analysis_df[numeric_columns].isna().any().any() or not np.isfinite(
        analysis_df[numeric_columns].to_numpy(dtype=float)
    ).all():
        raise ValueError("Required numeric result columns contain invalid values.")
    if (analysis_df["Best Possible Score"] <= 0).any():
        raise ValueError("Best Possible Score must be positive to calculate regret.")

    analysis_df["Simple Regret"] = (
        analysis_df["Best Possible Score"] - analysis_df[score_column]
    )
    analysis_df["Normalized Regret"] = (
        analysis_df["Simple Regret"] / analysis_df["Best Possible Score"]
    )
    analysis_df["Normalized Regret (%)"] = 100 * analysis_df["Normalized Regret"]

    methods = [method for method in METHOD_ORDER if method in analysis_df["Method"].unique()]
    missing_methods = [method for method in METHOD_ORDER if method not in methods]
    if missing_methods:
        raise ValueError(f"Expected all four methods; missing {missing_methods}.")

    def paired_wide(frame, budget):
        budget_frame = frame[frame["Iteration Budget"] == budget]
        if budget_frame.duplicated(["Profile ID", "Method"], keep=False).any():
            raise ValueError("Each profile-method-budget combination must have one result.")
        return budget_frame.pivot(
            index="Profile ID",
            columns="Method",
            values="Normalized Regret",
        ).reindex(columns=methods).dropna()

    def paired_sign_flip_test(first, second, seed):
        differences = np.asarray(first, dtype=float) - np.asarray(second, dtype=float)
        observed_difference = float(np.mean(differences))
        rng = np.random.default_rng(seed)
        signs = rng.choice(
            np.array([-1.0, 1.0]), size=(n_permutations, len(differences))
        )
        permuted_means = np.mean(signs * differences, axis=1)
        p_value = (
            1 + np.count_nonzero(np.abs(permuted_means) >= abs(observed_difference))
        ) / (n_permutations + 1)
        return differences, observed_difference, p_value

    def paired_bootstrap_ci(differences, seed):
        differences = np.asarray(differences, dtype=float)
        rng = np.random.default_rng(seed)
        indices = rng.integers(0, len(differences), size=(n_bootstrap, len(differences)))
        return tuple(np.quantile(differences[indices].mean(axis=1), [0.025, 0.975]))

    def omnibus_permutation_test(wide, seed):
        values = wide.to_numpy(dtype=float)
        observed_statistic = float(np.ptp(values.mean(axis=0)))
        rng = np.random.default_rng(seed)
        permuted_statistics = np.empty(n_permutations)
        for permutation_index in range(n_permutations):
            permuted_values = np.array(
                [rng.permutation(profile_values) for profile_values in values]
            )
            permuted_statistics[permutation_index] = np.ptp(
                permuted_values.mean(axis=0)
            )
        p_value = (
            1 + np.count_nonzero(permuted_statistics >= observed_statistic)
        ) / (n_permutations + 1)
        return observed_statistic, p_value

    paired_primary_metric = {}
    summary_rows = []
    omnibus_rows = []
    pairwise_rows = []
    skipped_budget_rows = []
    budgets = sorted(analysis_df["Iteration Budget"].unique())

    for budget in budgets:
        wide = paired_wide(analysis_df, budget)
        if len(wide) < 2:
            skipped_budget_rows.append(
                {
                    "Iteration Budget": budget,
                    "N Complete Paired Profiles": len(wide),
                    "Reason": "At least two complete paired profiles are required.",
                }
            )
            continue

        paired_primary_metric[budget] = wide
        permutation_statistic, permutation_p_value = omnibus_permutation_test(
            wide, random_seed + int(budget)
        )
        friedman_statistic, friedman_p_value = stats.friedmanchisquare(
            *[wide[method].to_numpy() for method in methods]
        )
        omnibus_rows.append(
            {
                "Iteration Budget": budget,
                "N Paired Profiles": len(wide),
                "Endpoint": "Normalized noise-free regret (%)",
                "Paired Permutation Range Statistic": 100 * permutation_statistic,
                "Paired Permutation P Value": permutation_p_value,
                "Friedman Statistic": friedman_statistic,
                "Friedman P Value": friedman_p_value,
            }
        )

        budget_frame = analysis_df[
            (analysis_df["Iteration Budget"] == budget)
            & analysis_df["Profile ID"].isin(wide.index)
        ]
        for method in methods:
            method_values = wide[method]
            method_rows = budget_frame[budget_frame["Method"] == method]
            selected_values = method_rows[score_column]
            summary_rows.append(
                {
                    "Iteration Budget": budget,
                    "Method": method,
                    "N Profiles": len(method_values),
                    "Mean Normalized Regret (%)": 100 * method_values.mean(),
                    "SD Normalized Regret (%)": 100 * method_values.std(ddof=1),
                    "Median Normalized Regret (%)": 100 * method_values.median(),
                    "IQR Normalized Regret (%)": 100 * (
                        method_values.quantile(0.75) - method_values.quantile(0.25)
                    ),
                    "Mean Noise-Free Score": selected_values.mean(),
                    "SD Noise-Free Score": selected_values.std(ddof=1),
                    "Median Noise-Free Score": selected_values.median(),
                    "Mean Simple Regret": method_rows["Simple Regret"].mean(),
                    "Median Simple Regret": method_rows["Simple Regret"].median(),
                    "P Regret <= Tolerance": (
                        method_rows["Normalized Regret"] <= regret_tolerance
                    ).mean(),
                }
            )

        for pair_index, (method_a, method_b) in enumerate(itertools.combinations(methods, 2)):
            differences, mean_difference, p_value = paired_sign_flip_test(
                wide[method_a],
                wide[method_b],
                random_seed + 100000 + int(budget) * 100 + pair_index,
            )
            ci_low, ci_high = paired_bootstrap_ci(
                differences,
                random_seed + 200000 + int(budget) * 100 + pair_index,
            )
            difference_sd = np.std(differences, ddof=1)
            pairwise_rows.append(
                {
                    "Iteration Budget": budget,
                    "Method A": method_a,
                    "Method B": method_b,
                    "N Paired Profiles": len(differences),
                    "Mean Normalized Regret Difference (A - B) (%)": 100 * mean_difference,
                    "95% Bootstrap CI Low (%)": 100 * ci_low,
                    "95% Bootstrap CI High (%)": 100 * ci_high,
                    "Paired Effect Size dz": (
                        mean_difference / difference_sd if difference_sd > 0 else np.nan
                    ),
                    "P A Lower Regret Than B": np.mean(differences < 0),
                    "Paired Sign-Flip P Value": p_value,
                }
            )

    if not paired_primary_metric:
        raise ValueError("No iteration budget has at least two complete paired profiles.")

    summary_df = pd.DataFrame(summary_rows)
    omnibus_df = pd.DataFrame(omnibus_rows)
    pairwise_df = pd.DataFrame(pairwise_rows)
    skipped_budgets_df = pd.DataFrame(skipped_budget_rows)

    if not pairwise_df.empty:
        pairwise_df["Holm Adjusted P Value"] = np.nan
        for budget in pairwise_df["Iteration Budget"].unique():
            budget_mask = pairwise_df["Iteration Budget"] == budget
            pairwise_df.loc[budget_mask, "Holm Adjusted P Value"] = multipletests(
                pairwise_df.loc[budget_mask, "Paired Sign-Flip P Value"], method="holm"
            )[1]

    if "Sec To Compute" in analysis_df:
        cost_df = (
            analysis_df.groupby(["Iteration Budget", "Method"])["Sec To Compute"]
            .agg(
                N_Profiles="count",
                Mean_Seconds="mean",
                SD_Seconds="std",
                Median_Seconds="median",
            )
            .reindex(
                pd.MultiIndex.from_product(
                    [budgets, methods], names=["Iteration Budget", "Method"]
                )
            )
            .reset_index()
        )
    else:
        cost_df = pd.DataFrame()

    if family_convergence is None:
        convergence_df = pd.DataFrame()
    elif isinstance(family_convergence, pd.DataFrame):
        convergence_df = family_convergence.copy()
    elif isinstance(family_convergence, (list, tuple)):
        if not all(isinstance(frame, pd.DataFrame) for frame in family_convergence):
            raise TypeError("Each convergence result must be a pandas DataFrame.")
        convergence_df = pd.concat(family_convergence, ignore_index=True)
    else:
        raise TypeError("family_convergence must be a DataFrame, a list of DataFrames, or None.")

    trajectory_summary_df = pd.DataFrame()
    common_random_number_df = pd.DataFrame()
    incomplete_trajectory_df = pd.DataFrame()
    trajectory_score_column = None
    if not convergence_df.empty:
        if "Surface Family" not in convergence_df:
            convergence_df["Surface Family"] = family_name
        if "Profile ID" not in convergence_df:
            profile_seed_column = "Profile Seed" if "Profile Seed" in convergence_df else "Seed"
            if profile_seed_column in convergence_df:
                convergence_df["Profile ID"] = (
                    convergence_df["Surface Family"].astype(str)
                    + "::"
                    + convergence_df[profile_seed_column].astype(str)
                )
        else:
            convergence_df["Profile ID"] = convergence_df["Profile ID"].astype(str)
        if "Iteration Budget" not in convergence_df:
            convergence_df["Iteration Budget"] = iteration_budget

        best_possible_by_profile = (
            analysis_df.drop_duplicates("Profile ID")
            .set_index("Profile ID")["Best Possible Score"]
        )
        trajectory_score_column = _add_incumbent_regret(
            convergence_df, best_possible_by_profile
        )

        if trajectory_score_column is not None:
            trajectory_summary_df = (
                convergence_df.groupby(
                    ["Surface Family", "Method", "Iteration Budget", "Query"]
                )[trajectory_score_column]
                .agg(
                    Median="median",
                    Q25=lambda values: values.quantile(0.25),
                    Q75=lambda values: values.quantile(0.75),
                )
                .reset_index()
            )
            profile_counts = (
                convergence_df.groupby(
                    ["Surface Family", "Method", "Iteration Budget", "Query"]
                )["Profile ID"]
                .nunique()
                .reset_index(name="N Profiles")
            )
            trajectory_summary_df = trajectory_summary_df.merge(
                profile_counts,
                on=["Surface Family", "Method", "Iteration Budget", "Query"],
                how="left",
            )
            # A query summarised over fewer profiles than the family total is a biased subset.
            expected_profiles = convergence_df["Profile ID"].nunique()
            trajectory_summary_df["Complete Coverage"] = (
                trajectory_summary_df["N Profiles"] == expected_profiles
            )
            incomplete_trajectory_df = trajectory_summary_df[
                ~trajectory_summary_df["Complete Coverage"]
            ][["Surface Family", "Method", "Iteration Budget", "Query", "N Profiles"]]
            incomplete_trajectory_df = incomplete_trajectory_df.assign(
                **{"N Profiles Expected": expected_profiles}
            ).reset_index(drop=True)

        crn_columns = {"Profile ID", "Iteration Budget", "Query", "Method", "Evaluation Seed"}
        if crn_columns.issubset(convergence_df.columns):
            common_random_number_df = (
                convergence_df.groupby(["Profile ID", "Iteration Budget", "Query"])
                .agg(
                    N_Methods=("Method", "nunique"),
                    N_Evaluation_Seeds=("Evaluation Seed", "nunique"),
                )
                .reset_index()
            )
            common_random_number_df["Uses Common Random Number"] = (
                (common_random_number_df["N_Methods"] == len(methods))
                & (common_random_number_df["N_Evaluation_Seeds"] == 1)
            )

    figures = {}
    if plot_results:
        primary_figure, primary_axes = plt.subplots(
            1, len(paired_primary_metric), figsize=(6 * len(paired_primary_metric), 5), squeeze=False
        )
        for axis, budget in zip(primary_axes.ravel(), sorted(paired_primary_metric)):
            wide = paired_primary_metric[budget]
            axis.boxplot(
                [100 * wide[method].to_numpy() for method in methods],
                labels=methods,
                patch_artist=True,
            )
            axis.set_title(f"{family_name}: normalized regret, {budget} queries")
            axis.set_ylabel("Normalized noise-free regret (%)")
            axis.tick_params(axis="x", labelrotation=25)
            axis.grid(axis="y", alpha=0.25)
        primary_figure.tight_layout()
        figures["normalized_regret_distribution"] = primary_figure
        plt.show()

        if not cost_df.empty:
            cost_figure, cost_axis = plt.subplots(figsize=(8, 5))
            cost_df.pivot(index="Method", columns="Iteration Budget", values="Mean_Seconds").reindex(
                methods
            ).plot(
                kind="bar",
                yerr=cost_df.pivot(
                    index="Method", columns="Iteration Budget", values="SD_Seconds"
                ).reindex(methods),
                ax=cost_axis,
                capsize=4,
            )
            cost_axis.set_title(f"{family_name}: computational cost")
            cost_axis.set_ylabel("Seconds per profile")
            cost_axis.grid(axis="y", alpha=0.25)
            cost_figure.tight_layout()
            figures["computational_cost"] = cost_figure
            plt.show()

        if not trajectory_summary_df.empty:
            trajectory_budgets = sorted(trajectory_summary_df["Iteration Budget"].unique())
            trajectory_figure, trajectory_axes = plt.subplots(
                1, len(trajectory_budgets), figsize=(7 * len(trajectory_budgets), 5), squeeze=False
            )
            for axis, budget in zip(trajectory_axes.ravel(), trajectory_budgets):
                budget_trajectory = trajectory_summary_df[
                    (trajectory_summary_df["Iteration Budget"] == budget)
                    & trajectory_summary_df["Complete Coverage"]
                ]
                for method in methods:
                    method_data = budget_trajectory[
                        budget_trajectory["Method"] == method
                    ].sort_values("Query")
                    if method_data.empty:
                        continue
                    line = axis.plot(
                        method_data["Query"], method_data["Median"], label=method
                    )[0]
                    axis.fill_between(
                        method_data["Query"],
                        method_data["Q25"],
                        method_data["Q75"],
                        alpha=0.15,
                        color=line.get_color(),
                    )
                    boundary = INITIAL_DESIGN_QUERIES.get(method)
                    if boundary is not None:
                        axis.axvline(
                            boundary,
                            color=line.get_color(),
                            linestyle=":",
                            alpha=0.7,
                            linewidth=1.2,
                        )
                axis.set_title(f"{family_name}: regret convergence, {budget} queries")
                axis.set_xlabel("Query")
                axis.set_ylabel("Normalized noise-free regret (%)")
                axis.grid(alpha=0.25)
            trajectory_figure.legend(loc="lower center", ncol=2)
            trajectory_figure.suptitle(
                "Regret of the parameters each optimizer would recommend after each query. "
                "Dotted lines mark the end of each initial design "
                "(BO 4, SMAC 10, TPE 20 queries; Random Search never switches to a model).",
                fontsize=9,
                y=0.99,
            )
            trajectory_figure.tight_layout(rect=(0, 0.08, 1, 0.95))
            figures["convergence"] = trajectory_figure
            plt.show()

    common_random_number_summary_df = _build_common_random_number_summary(
        common_random_number_df
    )
    trajectory_summary_compact_df = _build_convergence_summary(trajectory_summary_df)
    if display_results:
        print(
            "Primary endpoint: normalized noise-free regret (%); lower is better. "
            "Selected Noise-Free Score is retained as a secondary absolute metric."
        )
        if trajectory_score_column == INCUMBENT_REGRET_COLUMN:
            print(
                "Convergence endpoint: noise-free regret of the incumbent, i.e. the "
                "parameters the optimizer would recommend if stopped at that query."
            )
        else:
            print("No selected-incumbent convergence data were supplied.")
        print("\n--- Primary normalized regret and secondary score summaries ---")
        display(summary_df)
        print("\n--- Paired omnibus tests on normalized regret ---")
        display(omnibus_df)
        print("\n--- Paired sign-flip tests with Holm correction ---")
        display(pairwise_df)
        print("\n--- Computational cost ---")
        display(cost_df)
        if not common_random_number_df.empty:
            print("\n--- Common-random-number audit ---")
            display(common_random_number_summary_df)
        if not trajectory_summary_df.empty:
            print("\n--- Normalized regret convergence summary ---")
            display(trajectory_summary_compact_df)
        if not incomplete_trajectory_df.empty:
            print(
                "\n--- Queries excluded from the convergence figure (incomplete profile coverage) ---"
            )
            display(incomplete_trajectory_df)
        if not skipped_budgets_df.empty:
            print("\n--- Budgets omitted from inferential tests ---")
            display(skipped_budgets_df)

    return {
        "family_name": family_name,
        "score_column": "Normalized Regret",
        "primary_metric_column": "Normalized Regret",
        "selected_score_column": score_column,
        "regret_tolerance": regret_tolerance,
        "analysis_df": analysis_df,
        "paired_scores": paired_primary_metric,
        "paired_primary_metric": paired_primary_metric,
        "summary_df": summary_df,
        "omnibus_df": omnibus_df,
        "pairwise_df": pairwise_df,
        "cost_df": cost_df,
        "convergence_df": convergence_df,
        "trajectory_score_column": trajectory_score_column,
        "trajectory_summary_df": trajectory_summary_df,
        "trajectory_summary_compact_df": trajectory_summary_compact_df,
        "incomplete_trajectory_df": incomplete_trajectory_df,
        "common_random_number_df": common_random_number_df,
        "common_random_number_summary_df": common_random_number_summary_df,
        "skipped_budgets_df": skipped_budgets_df,
        "figures": figures,
    }


### Regret definitions and validation

- **Selected incumbent:** the noise-free regret of the configuration selected from observations available after that query. Bayesian optimization, TPE, and random search select the first configuration with the highest observed score; SMAC uses its intensifier's incumbent after the trial is reported.
- **Oracle best evaluated:** the regret of the best noise-free point evaluated so far. This diagnostic uses information unavailable to the optimizer and is never used to choose parameters.
- `Cumulative Best Observed Score` is the highest single noisy observation; it is descriptive and does not determine SMAC's incumbent.
- `Normalized Regret So Far (%)` is retained as a convergence alias for `Selected Incumbent Normalized Regret (%)`. The final table's existing `Normalized Regret (%)` is also selected-incumbent regret. Both tables include an explicitly named `Oracle Best-Evaluated Normalized Regret (%)` diagnostic. Both metrics use the notebook's existing `Best Possible Score` reference.
- Older convergence CSVs do not contain SMAC's actual per-query incumbent. Regenerate convergence data before using them for trajectory analysis; existing final-result tables can still be analyzed without convergence data.

The tests below use small, temporary validation runs and do not export research results. The local compatibility check used SMAC 2.4.0 with scikit-learn 1.7.2. SMAC 2.4.0 fails to import with scikit-learn 1.9.1; if a Colab runtime has that combination, choose a compatible scikit-learn version and restart the runtime, recording the dependency change with the experiment.


In [ ]:
# Focused validation of regret tracking; this does not export research results.
import contextlib
import io
import os
import tempfile
import unittest
import warnings
from types import SimpleNamespace
from unittest import mock
from sklearn.exceptions import ConvergenceWarning


def _regret_test_row(method, query, params, observed, true_score, oracle_score):
    return {
        "Profile ID": "validation::1",
        "Method": method,
        "Query": query,
        "Amplitude": params["amplitude"],
        "Frequency": params["frequency"],
        "Pulse Width": params["pulse_width"],
        "Observed Score": observed,
        "Noise-Free Score": true_score,
        "Best Noise-Free Score So Far": oracle_score,
        INCUMBENT_POLICY_COLUMN: (
            SMAC_INCUMBENT_POLICY if method == SMAC_METHOD else OBSERVED_INCUMBENT_POLICY
        ),
    }


class RegretTrackingTests(unittest.TestCase):
    p1 = {"amplitude": 1.0, "frequency": 10.0, "pulse_width": 100.0}
    p2 = {"amplitude": 2.0, "frequency": 20.0, "pulse_width": 200.0}
    p3 = {"amplitude": 3.0, "frequency": 30.0, "pulse_width": 300.0}

    def test_bayesian_parameter_transforms_preserve_physical_bounds(self):
        bounds = {
            "amplitude": (0.5, 5.0),
            "frequency": (1.0, 50.0),
            "pulse_width": (50.0, 600.0),
        }
        lower = {name: limits[0] for name, limits in bounds.items()}
        upper = {name: limits[1] for name, limits in bounds.items()}
        midpoint = {name: sum(limits) / 2 for name, limits in bounds.items()}

        normalized_lower = _normalize_parameters(lower, bounds)
        normalized_upper = _normalize_parameters(upper, bounds)
        normalized_midpoint = _normalize_parameters(midpoint, bounds)
        self.assertEqual(normalized_lower, {name: 0.0 for name in bounds})
        self.assertEqual(normalized_upper, {name: 1.0 for name in bounds})
        np.testing.assert_allclose(list(normalized_midpoint.values()), [0.5] * 3)
        self.assertEqual(_denormalize_parameters(normalized_lower, bounds), lower)
        self.assertEqual(_denormalize_parameters(normalized_upper, bounds), upper)
        self.assertEqual(_denormalize_parameters(normalized_midpoint, bounds), midpoint)
        self.assertEqual(BOGP_DEFAULTS["gp_length_scale"], 1.0)


    def test_model_based_optimizers_share_four_startup_points(self):
        self.assertEqual(OPTIMIZER_STARTUP_POINTS, 4)
        self.assertEqual(BOGP_DEFAULTS["init_points"], OPTIMIZER_STARTUP_POINTS)
        self.assertEqual(TPE_DEFAULTS["n_startup_jobs"], OPTIMIZER_STARTUP_POINTS)
        self.assertEqual(tpe.suggest.keywords["n_startup_jobs"], OPTIMIZER_STARTUP_POINTS)
        self.assertEqual(
            SMAC_DEFAULTS["initial_design"]["n_configs"],
            OPTIMIZER_STARTUP_POINTS,
        )
        self.assertEqual(SMAC_DEFAULTS["initial_design"]["max_ratio"], 1.0)
        self.assertNotIn(
            "n_configs_per_hyperparameter", SMAC_DEFAULTS["initial_design"]
        )


    def test_gp_noise_variances_follow_simulator_distributions(self):
        amplitudes = np.array([0.5, 2.0, 5.0])
        expected_variances = {
            "normal": np.array([4.0, 4.0, 4.0]),
            "heteroscedastic": np.array([0.25, 4.0, 25.0]),
            "heavy_tailed": np.array([12.0, 12.0, 12.0]),
            "none": np.array([0.0, 0.0, 0.0]),
        }
        targets = np.array([10.0, 14.0, 20.0, 26.0])
        for noise_type, expected in expected_variances.items():
            actual = _gp_observation_noise_variances(noise_type, amplitudes)
            np.testing.assert_allclose(actual, expected)
            normalized = _normalized_gp_alpha(actual, targets, normalize_y=True)
            np.testing.assert_allclose(normalized, expected / np.std(targets) ** 2)
            unnormalized = _normalized_gp_alpha(actual, targets, normalize_y=False)
            np.testing.assert_allclose(unnormalized, expected)
        self.assertEqual(baseline_family["noise"], "normal")
        self.assertEqual(BOGP_DEFAULTS["gp_alpha_original"], 1e-6)
        self.assertEqual(BOGP_DEFAULTS["gp_numerical_jitter"], 1e-6)
        self.assertTrue(BOGP_DEFAULTS["allow_duplicate_points"])


    def test_heteroscedastic_gp_uses_per_observation_alpha(self):
        physical_bounds = {
            "amplitude": (0.5, 5.0),
            "frequency": (1.0, 50.0),
            "pulse_width": (50.0, 600.0),
        }
        unit_bounds = {name: (0.0, 1.0) for name in physical_bounds}
        optimizer = BayesianOptimization(
            f=lambda amplitude, frequency, pulse_width: amplitude + frequency + pulse_width,
            pbounds=unit_bounds,
            gp_noise_type="heteroscedastic",
            gp_physical_bounds=physical_bounds,
            random_state=1,
            verbose=0,
        )
        configurations = [
            {"amplitude": 0.0, "frequency": 0.1, "pulse_width": 0.2},
            {"amplitude": 0.5, "frequency": 0.4, "pulse_width": 0.6},
            {"amplitude": 1.0, "frequency": 0.7, "pulse_width": 0.8},
        ]
        targets = np.array([10.0, 20.0, 35.0])
        for configuration, target in zip(configurations, targets):
            optimizer.register(params=configuration, target=target)

        optimizer.suggest()
        normalized_amplitudes = np.array(
            [configuration["amplitude"] for configuration in configurations]
        )
        lower, upper = physical_bounds["amplitude"]
        physical_amplitudes = lower + normalized_amplitudes * (upper - lower)
        expected_alpha = (
            physical_amplitudes ** 2 / np.std(targets) ** 2
            + BOGP_DEFAULTS["gp_numerical_jitter"]
        )
        np.testing.assert_allclose(optimizer._gp.alpha, expected_alpha)
        self.assertTrue(BOGP_DEFAULTS["gp_normalize_y"])
        np.testing.assert_allclose(np.ravel(optimizer._gp._y_train_std), [np.std(targets)])


    def _noisy_selection_rows(self, true_scores):
        rows = []
        best_observed, selected, best_true = -np.inf, None, -np.inf
        for query, (params, observed, true_score) in enumerate(
            zip((self.p1, self.p2, self.p3), (10.0, 11.0, 9.0), true_scores), 1
        ):
            best_observed, selected = _update_observed_incumbent(
                best_observed, selected, observed, params
            )
            best_true = max(best_true, true_score)
            row = _regret_test_row("Random Search", query, params, observed, true_score, best_true)
            _set_incumbent_parameters(row, selected)
            rows.append(row)
        _finalize_convergence_metrics(rows, 100.0)
        return rows

    def test_selected_and_oracle_are_distinct(self):
        rows = self._noisy_selection_rows((90.0, 20.0, 95.0))
        np.testing.assert_allclose(
            [row[SELECTED_INCUMBENT_REGRET_COLUMN] for row in rows], [10, 80, 80]
        )
        np.testing.assert_allclose(
            [row[ORACLE_BEST_EVALUATED_REGRET_COLUMN] for row in rows], [10, 10, 5]
        )
        self.assertEqual(rows[-1]["Normalized Regret So Far (%)"], 80.0)
        # Changing diagnostic scores cannot change decisions made from noisy scores.
        changed_truth = self._noisy_selection_rows((10.0, 99.0, 50.0))
        for original, changed in zip(rows, changed_truth):
            for column in INCUMBENT_PARAMETER_COLUMNS.values():
                self.assertEqual(original[column], changed[column])
        best, selected = _update_observed_incumbent(10.0, self.p1, 10.0, self.p2)
        self.assertEqual((best, selected), (10.0, self.p1))

    def test_smac_uses_intensifier_even_when_a_noisy_score_is_higher(self):
        recorder = SMACIncumbentRecorder()
        smbo = SimpleNamespace(intensifier=SimpleNamespace(get_incumbent=lambda: self.p1))
        recorder.on_tell_end(smbo, SimpleNamespace(config=self.p1), SimpleNamespace(cost=-5.0))
        recorder.on_tell_end(smbo, SimpleNamespace(config=self.p2), SimpleNamespace(cost=-99.0))
        rows = [
            _regret_test_row(SMAC_METHOD, 1, self.p1, 5.0, 70.0, 70.0),
            _regret_test_row(SMAC_METHOD, 2, self.p2, 99.0, 90.0, 90.0),
        ]
        _attach_smac_incumbents(rows, recorder.records, 2)
        _finalize_convergence_metrics(rows, 100.0)
        self.assertEqual(rows[-1]["Selected Incumbent Amplitude (mA)"], self.p1["amplitude"])
        self.assertEqual(rows[-1]["Current Incumbent Noise-Free Score"], 70.0)
        self.assertEqual(rows[-1][SELECTED_INCUMBENT_REGRET_COLUMN], 30.0)
        self.assertEqual(rows[-1][ORACLE_BEST_EVALUATED_REGRET_COLUMN], 10.0)
        with self.assertRaisesRegex(RuntimeError, "counts differ"):
            _attach_smac_incumbents(rows, recorder.records[:1], 2)
        bad_records = [dict(record) for record in recorder.records]
        bad_records[1]["cost"] = -98.0
        with self.assertRaisesRegex(RuntimeError, "do not match"):
            _attach_smac_incumbents(rows, bad_records, 2)

    def test_legacy_or_inconsistent_trajectories_are_rejected(self):
        frame = pd.DataFrame(self._noisy_selection_rows((90.0, 20.0, 95.0)))
        reference = pd.Series({"validation::1": 100.0})
        self.assertEqual(_add_incumbent_regret(frame.copy(), reference), INCUMBENT_REGRET_COLUMN)
        with self.assertRaisesRegex(ValueError, "older notebook"):
            _add_incumbent_regret(frame.drop(columns=[SELECTED_INCUMBENT_REGRET_COLUMN]), reference)
        bad_policy = frame.copy()
        bad_policy.loc[0, INCUMBENT_POLICY_COLUMN] = SMAC_INCUMBENT_POLICY
        with self.assertRaisesRegex(ValueError, "selection policy"):
            _add_incumbent_regret(bad_policy, reference)
        bad_regret = frame.copy()
        bad_regret.loc[0, SELECTED_INCUMBENT_REGRET_COLUMN] += 1
        with self.assertRaisesRegex(ValueError, "does not match"):
            _add_incumbent_regret(bad_regret, reference)
        bad_alias = frame.copy()
        bad_alias.loc[0, "Normalized Regret So Far (%)"] += 1
        with self.assertRaisesRegex(ValueError, "alias"):
            _add_incumbent_regret(bad_alias, reference)

    def test_unvisited_incumbent_and_final_mismatch_fail(self):
        row = _regret_test_row("Random Search", 1, self.p1, 5.0, 90.0, 90.0)
        _set_incumbent_parameters(row, self.p2)
        with self.assertRaisesRegex(RuntimeError, "not evaluated"):
            _finalize_convergence_metrics([row], 100.0)
        rows = self._noisy_selection_rows((90.0, 20.0, 95.0))
        result = pd.DataFrame([{
            "Method": "Random Search",
            "Selected Amplitude (mA)": self.p2["amplitude"],
            "Selected Frequency (Hz)": self.p2["frequency"],
            "Selected Pulse Width (us)": self.p2["pulse_width"],
            "Selected Score (Noise-Free)": 20.0,
            "Normalized Regret (%)": 80.0,
            SELECTED_INCUMBENT_REGRET_COLUMN: 80.0,
            ORACLE_BEST_EVALUATED_REGRET_COLUMN: 5.0,
        }])
        _validate_final_convergence(result, pd.DataFrame(rows))
        result.loc[0, "Normalized Regret (%)"] += 1
        with self.assertRaisesRegex(RuntimeError, "Final convergence"):
            _validate_final_convergence(result, pd.DataFrame(rows))

    def test_real_optimizers_use_noisy_feedback_and_match_final_results(self):
        feedback = {method: [] for method in ("Bayesian Optimization", "TPE Optimization", SMAC_METHOD)}
        smac_incumbents = []
        bo_runs = []
        original_bo = BayesianOptimization
        original_fmin = fmin
        original_smac = HyperparameterOptimizationFacade
        original_recorder = SMACIncumbentRecorder

        class FeedbackBO(original_bo):
            def __init__(self, *args, **kwargs):
                objective = kwargs["f"]
                bo_run = {
                    "bounds": dict(kwargs["pbounds"]),
                    "noise_type": kwargs["gp_noise_type"],
                    "proposals": [],
                    "fit_records": [],
                    "maximize_settings": None,
                }
                bo_runs.append(bo_run)
                def observed_objective(amplitude, frequency, pulse_width):
                    bo_run["proposals"].append({
                        "amplitude": amplitude,
                        "frequency": frequency,
                        "pulse_width": pulse_width,
                    })
                    value = objective(amplitude, frequency, pulse_width)
                    feedback["Bayesian Optimization"].append(float(value))
                    return value
                kwargs["f"] = observed_objective
                super().__init__(*args, **kwargs)
                self._bo_run = bo_run
                bo_run["gp_length_scale"] = float(self._gp.kernel.length_scale)

            def suggest(self):
                suggestion = super().suggest()
                self._bo_run["fit_records"].append((
                    np.asarray(self._gp.alpha, dtype=float).copy(),
                    np.asarray(self._space.target, dtype=float).copy(),
                ))
                return suggestion

            def maximize(self, *args, **kwargs):
                self._bo_run["maximize_settings"] = dict(kwargs)
                return super().maximize(*args, **kwargs)

        def feedback_fmin(*args, **kwargs):
            objective = kwargs["fn"]
            def observed_objective(params):
                result = objective(params)
                feedback["TPE Optimization"].append(-float(result["loss"]))
                return result
            kwargs["fn"] = observed_objective
            return original_fmin(*args, **kwargs)

        class FeedbackSMAC(original_smac):
            def __init__(self, scenario, target_function=None, **kwargs):
                def observed_objective(config, seed=None):
                    cost = target_function(config, seed=seed)
                    feedback[SMAC_METHOD].append(-float(cost))
                    return cost
                super().__init__(scenario, observed_objective, **kwargs)

        class CheckedRecorder(original_recorder):
            def on_tell_end(self, smbo, info, value):
                super().on_tell_end(smbo, info, value)
                smac_incumbents.append(_physical_parameters(smbo.intensifier.get_incumbent()))

        original_directory = Path.cwd().resolve()
        with tempfile.TemporaryDirectory(prefix="regret-validation-", dir=original_directory) as directory:
            test_directory = Path(directory).resolve()
            self.assertEqual(test_directory.parent, original_directory)
            try:
                os.chdir(test_directory)
                with mock.patch.dict(globals(), {
                    "BayesianOptimization": FeedbackBO,
                    "fmin": feedback_fmin,
                    "HyperparameterOptimizationFacade": FeedbackSMAC,
                    "SMACIncumbentRecorder": CheckedRecorder,
                }), contextlib.redirect_stdout(io.StringIO()), warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=ConvergenceWarning)
                    results, traces = run_optimization_comparison(12, 2, 42091, baseline_family)
            finally:
                os.chdir(original_directory)

        for patient_results, patient_trace in zip(results, traces):
            _validate_final_convergence(patient_results, patient_trace)
        physical_bounds = {
            "amplitude": (0.5, 5.0),
            "frequency": (1.0, 50.0),
            "pulse_width": (50.0, 600.0),
        }
        expected_bo_bounds = {name: (0.0, 1.0) for name in physical_bounds}
        self.assertEqual(len(bo_runs), len(traces))
        for bo_run, patient_trace in zip(bo_runs, traces):
            self.assertEqual(
                bo_run["maximize_settings"]["init_points"],
                OPTIMIZER_STARTUP_POINTS,
            )
            self.assertEqual(
                bo_run["maximize_settings"]["n_iter"],
                12 - OPTIMIZER_STARTUP_POINTS,
            )
            self.assertEqual(bo_run["bounds"], expected_bo_bounds)
            self.assertEqual(bo_run["noise_type"], "normal")
            self.assertEqual(bo_run["gp_length_scale"], BOGP_DEFAULTS["gp_length_scale"])
            for alpha, observed_targets in bo_run["fit_records"]:
                noise_variances = _gp_observation_noise_variances(
                    "normal", np.zeros_like(observed_targets)
                )
                expected_alpha = _normalized_gp_alpha(
                    noise_variances,
                    observed_targets,
                    BOGP_DEFAULTS["gp_normalize_y"],
                ) + BOGP_DEFAULTS["gp_numerical_jitter"]
                np.testing.assert_allclose(alpha, expected_alpha)
            bo_rows = patient_trace[
                patient_trace["Method"] == "Bayesian Optimization"
            ].sort_values("Query")
            proposals = bo_run["proposals"]
            self.assertEqual(len(bo_rows), len(proposals))
            for (_, row), normalized_params in zip(bo_rows.iterrows(), proposals):
                self.assertTrue(all(0.0 <= value <= 1.0 for value in normalized_params.values()))
                recorded_params = {
                    "amplitude": row["Amplitude"],
                    "frequency": row["Frequency"],
                    "pulse_width": row["Pulse Width"],
                }
                expected_params = _denormalize_parameters(normalized_params, physical_bounds)
                np.testing.assert_allclose(
                    list(recorded_params.values()),
                    list(expected_params.values()),
                    rtol=0,
                    atol=1e-12,
                )
        result = pd.concat(results, ignore_index=True)
        trace = pd.concat(traces, ignore_index=True)
        np.testing.assert_allclose(result["Normalized Regret (%)"], result[SELECTED_INCUMBENT_REGRET_COLUMN])
        reference = (
            result.assign(**{"Profile ID": result["Surface Family"] + "::" + result["Seed"].astype(str)})
            .drop_duplicates("Profile ID").set_index("Profile ID")["Best Possible Score"]
        )
        self.assertEqual(_add_incumbent_regret(trace.copy(), reference), INCUMBENT_REGRET_COLUMN)
        self.assertEqual(set(trace["Method"]), set(METHOD_ORDER))
        np.testing.assert_allclose(
            trace["Normalized Regret So Far (%)"], trace[SELECTED_INCUMBENT_REGRET_COLUMN]
        )
        for method, observed_feedback in feedback.items():
            method_rows = trace[trace["Method"] == method]
            np.testing.assert_allclose(observed_feedback, method_rows["Observed Score"])
            self.assertTrue(np.any(~np.isclose(method_rows["Observed Score"], method_rows["Noise-Free Score"])))
        smac_rows = trace[trace["Method"] == SMAC_METHOD]
        self.assertEqual(len(smac_incumbents), len(smac_rows))
        expected_smac = [[item[name] for name in INCUMBENT_PARAMETER_COLUMNS] for item in smac_incumbents]
        np.testing.assert_allclose(smac_rows[list(INCUMBENT_PARAMETER_COLUMNS.values())], expected_smac)
        for (profile, method), group in trace.groupby(["Profile ID", "Method"]):
            group = group.sort_values("Query").reset_index(drop=True)
            self.assertEqual(len(group), 12)
            oracle = group[ORACLE_BEST_EVALUATED_REGRET_COLUMN].to_numpy()
            selected_regret = group[SELECTED_INCUMBENT_REGRET_COLUMN].to_numpy()
            self.assertTrue(np.all(np.diff(oracle) <= 1e-10))
            self.assertTrue(np.all(oracle <= selected_regret + 1e-10))
            if method != SMAC_METHOD:
                expected = []
                for query in range(1, len(group) + 1):
                    best_index = int(np.argmax(group["Observed Score"].to_numpy()[:query]))
                    best_row = group.iloc[best_index]
                    expected.append([best_row["Amplitude"], best_row["Frequency"], best_row["Pulse Width"]])
                np.testing.assert_allclose(group[list(INCUMBENT_PARAMETER_COLUMNS.values())], expected)

        analysis_options = dict(
            iteration_budget=12, n_permutations=10, n_bootstrap=10,
            display_results=False, plot_results=False,
        )
        analysis = analyze_family_results(results, traces, **analysis_options)
        self.assertEqual(analysis["trajectory_score_column"], SELECTED_INCUMBENT_REGRET_COLUMN)
        final_medians = result.groupby("Method")["Normalized Regret (%)"].median()
        trajectory_medians = (
            analysis["trajectory_summary_df"].query("Query == 12").set_index("Method")["Median"]
        )
        np.testing.assert_allclose(final_medians.reindex(METHOD_ORDER), trajectory_medians.reindex(METHOD_ORDER))
        legacy = [frame.drop(columns=[SELECTED_INCUMBENT_REGRET_COLUMN]) for frame in traces]
        with self.assertRaisesRegex(ValueError, "older notebook"):
            analyze_family_results(results, legacy, **analysis_options)
        final_only = analyze_family_results(results, **analysis_options)
        self.assertIsNone(final_only["trajectory_score_column"])


    def test_full_budget_smac_incumbent_bounds_and_reproducibility(self):
        seed = 73191
        original_smac = HyperparameterOptimizationFacade
        smac_runs = []

        class AuditedSMAC(original_smac):
            def __init__(self, scenario, target_function=None, **kwargs):
                self._audit_scenario = scenario
                super().__init__(scenario, target_function, **kwargs)

            def optimize(self, *args, **kwargs):
                incumbent = super().optimize(*args, **kwargs)
                scenario = self._audit_scenario
                smac_runs.append({
                    "finished": self.runhistory.finished,
                    "initial_design_points": self._initial_design._n_configs,
                    "scenario": {
                        "deterministic": scenario.deterministic,
                        "objectives": scenario.objectives,
                        "use_default_config": scenario.use_default_config,
                        "n_workers": scenario.n_workers,
                        "n_trials": scenario.n_trials,
                        "seed": scenario.seed,
                    },
                    "incumbent": _physical_parameters(incumbent),
                    "incumbent_cost": self.runhistory.get_cost(incumbent),
                })
                return incumbent

        original_directory = Path.cwd().resolve()

        def run_once():
            with tempfile.TemporaryDirectory(
                prefix="full-budget-validation-", dir=original_directory
            ) as directory:
                test_directory = Path(directory).resolve()
                self.assertEqual(test_directory.parent, original_directory)
                try:
                    os.chdir(test_directory)
                    with mock.patch.dict(globals(), {
                        "HyperparameterOptimizationFacade": AuditedSMAC,
                    }), contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()), warnings.catch_warnings():
                        warnings.simplefilter("ignore", category=ConvergenceWarning)
                        results, traces = run_optimization_comparison(
                            40, 1, seed, baseline_family
                        )
                finally:
                    os.chdir(original_directory)
            return results[0], traces[0]

        results_a, trace_a = run_once()
        results_b, trace_b = run_once()
        self.assertEqual(len(smac_runs), 2)

        parameter_bounds = {
            "Amplitude": (0.5, 5.0),
            "Frequency": (1.0, 50.0),
            "Pulse Width": (50.0, 600.0),
        }
        selected_bounds = {
            "Selected Amplitude (mA)": parameter_bounds["Amplitude"],
            "Selected Frequency (Hz)": parameter_bounds["Frequency"],
            "Selected Pulse Width (us)": parameter_bounds["Pulse Width"],
        }
        for run_results, run_trace in ((results_a, trace_a), (results_b, trace_b)):
            self.assertEqual(set(run_trace["Iteration Budget"]), {40})
            self.assertEqual(set(run_trace["Method"]), set(METHOD_ORDER))
            for method in METHOD_ORDER:
                group = run_trace[run_trace["Method"] == method].sort_values("Query")
                self.assertEqual(len(group), 40, method)
                self.assertEqual(group["Query"].tolist(), list(range(1, 41)), method)
                expected_policy = (
                    SMAC_INCUMBENT_POLICY if method == SMAC_METHOD
                    else OBSERVED_INCUMBENT_POLICY
                )
                self.assertEqual(set(group[INCUMBENT_POLICY_COLUMN]), {expected_policy})
                for column, (lower, upper) in parameter_bounds.items():
                    self.assertTrue(group[column].between(lower, upper).all(), (method, column))
            evaluation_seed_sequences = [
                run_trace[run_trace["Method"] == method]
                .sort_values("Query")["Evaluation Seed"].tolist()
                for method in METHOD_ORDER
            ]
            for sequence in evaluation_seed_sequences[1:]:
                self.assertEqual(sequence, evaluation_seed_sequences[0])
            for column, (lower, upper) in selected_bounds.items():
                self.assertTrue(run_results[column].between(lower, upper).all(), column)

        expected_scenario = {
            "deterministic": SMAC_DEFAULTS["scenario"]["deterministic"],
            "objectives": SMAC_DEFAULTS["scenario"]["objectives"],
            "use_default_config": SMAC_DEFAULTS["scenario"]["use_default_config"],
            "n_workers": SMAC_DEFAULTS["scenario"]["n_workers"],
            "n_trials": 40,
            "seed": seed,
        }
        for run, run_trace in zip(smac_runs, (trace_a, trace_b)):
            self.assertEqual(run["scenario"], expected_scenario)
            self.assertEqual(run["finished"], 40)
            self.assertEqual(
                run["initial_design_points"], OPTIMIZER_STARTUP_POINTS
            )
            smac_rows = run_trace[
                run_trace["Method"] == SMAC_METHOD
            ].sort_values("Query").reset_index(drop=True)
            self.assertEqual(len(smac_rows), run["finished"])
            for query in range(1, len(smac_rows) + 1):
                current = smac_rows.iloc[query - 1]
                selected_params = current[list(INCUMBENT_PARAMETER_COLUMNS.values())].to_numpy(dtype=float)
                evaluated_params = smac_rows.iloc[:query][
                    ["Amplitude", "Frequency", "Pulse Width"]
                ].to_numpy(dtype=float)
                matching = np.all(
                    np.isclose(evaluated_params, selected_params, rtol=0, atol=1e-12),
                    axis=1,
                )
                self.assertTrue(np.any(matching), f"SMAC incumbent at query {query} was not evaluated.")
                matching_scores = smac_rows.iloc[:query].loc[matching, "Noise-Free Score"]
                self.assertAlmostEqual(
                    current["Current Incumbent Noise-Free Score"],
                    matching_scores.iloc[-1],
                )

        stable_result_columns = [
            column for column in results_a.columns if column != "Sec To Compute"
        ]
        pd.testing.assert_frame_equal(
            results_a[stable_result_columns],
            results_b[stable_result_columns],
            check_exact=False,
            rtol=0,
            atol=1e-12,
        )
        pd.testing.assert_frame_equal(
            trace_a, trace_b, check_exact=False, rtol=0, atol=1e-12
        )

        smac_result = results_a[results_a["Method"] == SMAC_METHOD].iloc[0]
        final_smac_row = (
            trace_a[trace_a["Method"] == SMAC_METHOD]
            .sort_values("Query")
            .iloc[-1]
        )
        result_parameter_columns = {
            "amplitude": "Selected Amplitude (mA)",
            "frequency": "Selected Frequency (Hz)",
            "pulse_width": "Selected Pulse Width (us)",
        }
        for parameter, result_column in result_parameter_columns.items():
            self.assertAlmostEqual(smac_runs[0]["incumbent"][parameter], smac_result[result_column])
            self.assertAlmostEqual(
                final_smac_row[INCUMBENT_PARAMETER_COLUMNS[parameter]],
                smac_result[result_column],
            )
        self.assertAlmostEqual(
            -smac_runs[0]["incumbent_cost"], smac_result["Best Computed Score"]
        )

_regret_test_suite = unittest.defaultTestLoader.loadTestsFromTestCase(RegretTrackingTests)
_regret_test_result = unittest.TextTestRunner(verbosity=2).run(_regret_test_suite)
if not _regret_test_result.wasSuccessful():
    raise AssertionError("Regret tracking validation failed; inspect the test output above.")


**WHERE ALL THE FAMILIES ARE RUN**

In [ ]:
baseline_family_outputs = run_and_export_family(
    family=baseline_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(baseline_family_outputs)["summary_df"]

In [ ]:
symmetric_family_outputs = run_and_export_family(
    family=symmetric_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(symmetric_family_outputs)["summary_df"]

In [ ]:
slow_rise_quick_fall_family_outputs = run_and_export_family(
    family=slow_rise_quick_fall_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(slow_rise_quick_fall_family_outputs)["summary_df"]

In [ ]:
three_optima_family_outputs = run_and_export_family(
    family=three_optima_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(three_optima_family_outputs)["summary_df"]

In [ ]:
broad_peaks_family_outputs = run_and_export_family(
    family=broad_peaks_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(broad_peaks_family_outputs)["summary_df"]

In [ ]:
sharp_peaks_family_outputs = run_and_export_family(
    family=sharp_peaks_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(sharp_peaks_family_outputs)["summary_df"]

In [ ]:
hetero_peaks_family_outputs = run_and_export_family(
    family=heteroscedastic_noise_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(hetero_peaks_family_outputs)["summary_df"]

In [ ]:
heavy_tailed_noise_family_outputs = run_and_export_family(
    family=heavy_tailed_noise_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(heavy_tailed_noise_family_outputs)["summary_df"]

In [ ]:
unique_parameter_interaction_family_outputs = run_and_export_family(
    family=unique_parameter_interaction_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(unique_parameter_interaction_family_outputs)["summary_df"]

In [ ]:

# Optional: skip rerunning the optimizer families by reloading previously exported CSVs.
# Only these two files per family are needed:
#   <family>_family_results.csv and <family>_family_convergence.csv

LOADED_FAMILY_SPECS = [
    {"family_name": "quick_rise_slow_fall", "outputs_var": "baseline_family_outputs"},
    {"family_name": "symmetric", "outputs_var": "symmetric_family_outputs"},
    {"family_name": "slow_rise_quick_fall", "outputs_var": "slow_rise_quick_fall_family_outputs"},
    {"family_name": "3optima", "outputs_var": "three_optima_family_outputs"},
    {"family_name": "broad_optima", "outputs_var": "broad_peaks_family_outputs"},
    {"family_name": "sharp_optima", "outputs_var": "sharp_peaks_family_outputs"},
    {"family_name": "heteroscedastic_noise", "outputs_var": "hetero_peaks_family_outputs"},
    {"family_name": "heavy_tailed_noise", "outputs_var": "heavy_tailed_noise_family_outputs"},
    {"family_name": "unique_parameter_interaction", "outputs_var": "unique_parameter_interaction_family_outputs"},
]

USE_INTERACTIVE_UPLOAD = True
FAMILY_CSV_DIR = "."


def upload_family_csvs(upload_dir="uploaded_family_csvs"):
    """Prompt for CSV uploads in Colab/Jupyter widgets; returns the directory holding them."""
    upload_dir = Path(upload_dir)
    upload_dir.mkdir(parents=True, exist_ok=True)

    try:
        from google.colab import files as colab_files
    except ImportError:
        colab_files = None

    if colab_files is not None:
        print("Select every *_family_results.csv and *_family_convergence.csv file to upload.")
        uploaded = colab_files.upload()
        for filename, contents in uploaded.items():
            # Colab appends " (1)", " (2)", etc. to duplicate filenames already
            # present in the runtime; strip that suffix to restore the canonical
            # "<family>_family_results.csv" / "<family>_family_convergence.csv" name.
            clean_name = re.sub(r" \(\d+\)(?=\.[^.]+$)", "", Path(filename).name)
            (upload_dir / clean_name).write_bytes(contents)
        print(f"Saved {len(uploaded)} uploaded file(s) to {upload_dir.resolve()}")
        return upload_dir

    print(
        "Not running in Colab. Copy the exported CSV files into "
        f"{upload_dir.resolve()} (or pass the folder that already contains them "
        "to load_family_outputs_from_csv)."
    )
    return upload_dir


def load_family_outputs_from_csv(
    input_dir=".",
    family_specs=None,
    iteration_budget=40,
    assign_globals=True,
    require_all=True,
):
    """Rebuild each family's outputs dictionary from previously exported CSV files."""
    input_dir = Path(input_dir)
    if not input_dir.is_dir():
        raise FileNotFoundError(f"{input_dir.resolve()} is not a directory.")

    family_specs = family_specs or LOADED_FAMILY_SPECS
    loaded_outputs = {}
    missing_files = []

    for spec in family_specs:
        family_name = spec["family_name"]
        safe_prefix = re.sub(r"[^A-Za-z0-9_-]+", "_", family_name).strip("_")
        results_path = input_dir / f"{safe_prefix}_family_results.csv"
        convergence_path = input_dir / f"{safe_prefix}_family_convergence.csv"

        absent = [path.name for path in (results_path, convergence_path) if not path.exists()]
        if absent:
            missing_files.extend(absent)
            continue

        results_df = pd.read_csv(results_path)
        convergence_df = pd.read_csv(convergence_path)

        if "Surface Family" not in results_df:
            results_df["Surface Family"] = family_name
        if "Surface Family" not in convergence_df:
            convergence_df["Surface Family"] = family_name
        for frame in (results_df, convergence_df):
            if "Iteration Budget" not in frame:
                frame["Iteration Budget"] = iteration_budget

        outputs = {
            "family_name": family_name,
            "results_df": results_df,
            "convergence_df": convergence_df,
            "source_files": {
                "results": str(results_path),
                "convergence": str(convergence_path),
            },
        }
        loaded_outputs[family_name] = outputs
        if assign_globals:
            globals()[spec["outputs_var"]] = outputs

        print(
            f"Loaded {family_name!r}: {len(results_df):,} profile-method rows, "
            f"{len(convergence_df):,} query rows."
        )

    if missing_files:
        message = (
            "Missing exported CSV files: "
            + ", ".join(missing_files)
            + f" (searched {input_dir.resolve()})."
        )
        if require_all:
            raise FileNotFoundError(message)
        print(message)

    return loaded_outputs


# Set USE_INTERACTIVE_UPLOAD = True in Colab to pick the files with a file dialog.
family_csv_dir = upload_family_csvs() if USE_INTERACTIVE_UPLOAD else FAMILY_CSV_DIR
loaded_family_outputs = load_family_outputs_from_csv(family_csv_dir)


In [ ]:
# Reload and analyze one family without rerunning any optimizer.
# Upload exactly 2 CSVs: one results file and one convergence file, from any family.

import io

SINGLE_FAMILY_USE_INTERACTIVE_UPLOAD = True
SINGLE_FAMILY_RESULTS_CSV = "family_results.csv"
SINGLE_FAMILY_CONVERGENCE_CSV = "family_convergence.csv"


def upload_two_family_csvs():
    """Upload exactly 2 files (any names) and split them into results vs convergence by column contents."""
    try:
        from google.colab import files as colab_files
    except ImportError as error:
        raise RuntimeError(
            "Interactive upload requires Google Colab. Set "
            "SINGLE_FAMILY_USE_INTERACTIVE_UPLOAD = False and point "
            "SINGLE_FAMILY_RESULTS_CSV / SINGLE_FAMILY_CONVERGENCE_CSV at local files."
        ) from error

    print("Select exactly 2 files: one results CSV and one convergence CSV.")
    uploaded = colab_files.upload()
    if len(uploaded) != 2:
        raise ValueError(f"Expected exactly 2 files; got {len(uploaded)}.")

    frames = {
        filename: pd.read_csv(io.BytesIO(contents))
        for filename, contents in uploaded.items()
    }

    results_df = convergence_df = None
    for frame in frames.values():
        if "Best Possible Score" in frame.columns:
            results_df = frame
        elif "Query" in frame.columns:
            convergence_df = frame

    if results_df is None or convergence_df is None:
        raise ValueError(
            "Could not tell which upload was the results file and which was "
            "the convergence file. The results file needs a 'Best Possible "
            "Score' column; the convergence file needs a 'Query' column."
        )

    return results_df, convergence_df


def analyze_uploaded_family(results_df, convergence_df, **analyze_kwargs):
    """Run analyze_family_results directly on two already-loaded DataFrames."""
    family_name = str(results_df["Surface Family"].iloc[0])
    outputs = {
        "family_name": family_name,
        "results_df": results_df,
        "convergence_df": convergence_df,
    }
    return outputs, analyze_family_results(outputs, family_name=family_name, **analyze_kwargs)


if SINGLE_FAMILY_USE_INTERACTIVE_UPLOAD:
    single_family_results_df, single_family_convergence_df = upload_two_family_csvs()
else:
    single_family_results_df = pd.read_csv(SINGLE_FAMILY_RESULTS_CSV)
    single_family_convergence_df = pd.read_csv(SINGLE_FAMILY_CONVERGENCE_CSV)

single_family_outputs, single_family_analysis = analyze_uploaded_family(
    single_family_results_df,
    single_family_convergence_df,
)
single_family_analysis["summary_df"]


In [ ]:
sensitivity_methods = [
    "Bayesian Optimization",
    "TPE Optimization",
    "SMAC Optimization",
    "Random Search",
]
sensitivity_regret_tolerance = 0.05
sensitivity_n_permutations = 20000
sensitivity_n_bootstrap = 10000
sensitivity_random_seed = 10001


sensitivity_family_specs = [
    {
        "family_name": "quick_rise_slow_fall",
        "outputs_var": "baseline_family_outputs",
    },
    {
        "family_name": "symmetric",
        "outputs_var": "symmetric_family_outputs",
    },
    {
        "family_name": "slow_rise_quick_fall",
        "outputs_var": "slow_rise_quick_fall_family_outputs",
    },
    {
        "family_name": "3optima",
        "outputs_var": "three_optima_family_outputs",
    },
    {
        "family_name": "broad_optima",
        "outputs_var": "broad_peaks_family_outputs",
    },
    {
        "family_name": "sharp_optima",
        "outputs_var": "sharp_peaks_family_outputs",
    },
    {
        "family_name": "heteroscedastic_noise",
        "outputs_var": "hetero_peaks_family_outputs",
    },
    {
        "family_name": "heavy_tailed_noise",
        "outputs_var": "heavy_tailed_noise_family_outputs",
    },
    {
        "family_name": "unique_parameter_interaction",
        "outputs_var": "unique_parameter_interaction_family_outputs",
    },
]


required_sensitivity_variables = []
for sensitivity_spec in sensitivity_family_specs:
    required_sensitivity_variables.extend(
        variable_name
        for variable_name in [
            sensitivity_spec.get("results_var"),
            sensitivity_spec.get("convergence_var"),
            sensitivity_spec.get("outputs_var"),
        ]
        if variable_name is not None
    )
missing_sensitivity_variables = [
    variable_name
    for variable_name in required_sensitivity_variables
    if variable_name not in globals()
]
if missing_sensitivity_variables:
    raise RuntimeError(
        "Run the formal family simulation cells before the final sensitivity "
        "analysis. Missing variables: "
        + ", ".join(missing_sensitivity_variables)
    )


def _concat_sensitivity_frames(data, label):
    if isinstance(data, pd.DataFrame):
        return data.copy()
    if isinstance(data, (list, tuple)) and data:
        if not all(isinstance(frame, pd.DataFrame) for frame in data):
            raise TypeError(f"Every item in {label} must be a pandas DataFrame.")
        return pd.concat([frame.copy() for frame in data], ignore_index=True)
    raise TypeError(f"{label} must be a non-empty DataFrame or list of DataFrames.")


def _unique_budget_values(frame):
    if frame.empty or "Iteration Budget" not in frame.columns:
        return []
    values = pd.to_numeric(frame["Iteration Budget"], errors="coerce")
    if values.isna().any():
        raise ValueError("Iteration Budget contains missing or non-numeric values.")
    return sorted(values.astype(int).unique().tolist())


def _get_sensitivity_sources(sensitivity_spec):
    if "outputs_var" in sensitivity_spec:
        outputs = globals()[sensitivity_spec["outputs_var"]]
        if not isinstance(outputs, dict):
            raise TypeError(
                f"{sensitivity_spec['outputs_var']} must be the dictionary "
                "returned by run_and_export_family."
            )
        results_data = outputs.get("results_df")
        if results_data is None:
            results_data = outputs.get("family_test")
        convergence_data = outputs.get("convergence_df")
        if convergence_data is None:
            convergence_data = outputs.get("family_convergence")
    else:
        results_data = globals()[sensitivity_spec["results_var"]]
        convergence_data = globals()[sensitivity_spec["convergence_var"]]
    return results_data, convergence_data


def _normalize_sensitivity_family(family_name, results_data, convergence_data):
    results_df = _concat_sensitivity_frames(results_data, f"{family_name} results")
    convergence_df = _concat_sensitivity_frames(
        convergence_data,
        f"{family_name} convergence",
    )

    for frame, label in [
        (results_df, f"{family_name} results"),
        (convergence_df, f"{family_name} convergence"),
    ]:
        if "Surface Family" not in frame.columns:
            frame["Surface Family"] = family_name
        frame["Surface Family"] = frame["Surface Family"].astype(str)
        observed_families = set(frame["Surface Family"].unique())
        if observed_families != {family_name}:
            raise ValueError(
                f"{label} contains {sorted(observed_families)}, expected "
                f"only {family_name!r}."
            )

    if "Seed" not in results_df.columns:
        raise ValueError(f"{family_name} results must contain a Seed column.")
    results_df["Profile ID"] = (
        family_name + "::" + results_df["Seed"].astype(str)
    )

    profile_seed_column = (
        "Profile Seed" if "Profile Seed" in convergence_df.columns else "Seed"
    )
    if profile_seed_column in convergence_df.columns:
        convergence_df["Profile ID"] = (
            family_name + "::" + convergence_df[profile_seed_column].astype(str)
        )
    elif "Profile ID" in convergence_df.columns:
        convergence_df["Profile ID"] = convergence_df["Profile ID"].astype(str)
    else:
        raise ValueError(
            f"{family_name} convergence must contain Profile ID, Profile Seed, or Seed."
        )

    budget_values = sorted(
        set(_unique_budget_values(results_df))
        | set(_unique_budget_values(convergence_df))
    )
    if len(budget_values) != 1:
        raise ValueError(
            f"{family_name} must have exactly one iteration budget; found "
            f"{budget_values}."
        )
    iteration_budget = budget_values[0]
    results_df["Iteration Budget"] = iteration_budget
    convergence_df["Iteration Budget"] = iteration_budget

    required_result_columns = {
        "Method",
        "Best Possible Score",
        "Selected Score (Noise-Free)",
    }
    missing_columns = required_result_columns.difference(results_df.columns)
    if missing_columns:
        raise ValueError(
            f"{family_name} results are missing {sorted(missing_columns)}."
        )
    results_df["Method"] = results_df["Method"].astype(str)
    observed_methods = set(results_df["Method"].unique())
    if observed_methods != set(sensitivity_methods):
        raise ValueError(
            f"{family_name} must contain exactly the four expected methods; "
            f"found {sorted(observed_methods)}."
        )

    for column in ["Best Possible Score", "Selected Score (Noise-Free)"]:
        results_df[column] = pd.to_numeric(results_df[column], errors="coerce")
    if results_df[["Best Possible Score", "Selected Score (Noise-Free)"]].isna().any().any():
        raise ValueError(f"{family_name} contains missing or non-numeric scores.")
    if not np.isfinite(
        results_df[["Best Possible Score", "Selected Score (Noise-Free)"]]
        .to_numpy(dtype=float)
    ).all():
        raise ValueError(f"{family_name} contains non-finite scores.")
    if (results_df["Best Possible Score"] <= 0).any():
        raise ValueError(f"{family_name} has non-positive Best Possible Score values.")

    duplicate_keys = results_df.duplicated(
        subset=["Profile ID", "Iteration Budget", "Method"],
        keep=False,
    )
    if duplicate_keys.any():
        raise ValueError(
            f"{family_name} has duplicate profile-method-budget result rows."
        )
    profile_method_counts = results_df.groupby(
        ["Profile ID", "Iteration Budget"]
    )["Method"].nunique()
    if (profile_method_counts != len(sensitivity_methods)).any():
        raise ValueError(
            f"{family_name} contains incomplete paired profiles; every profile "
            "must have all four methods."
        )

    results_df["Simple Regret"] = (
        results_df["Best Possible Score"]
        - results_df["Selected Score (Noise-Free)"]
    )
    results_df["Normalized Regret"] = (
        results_df["Simple Regret"] / results_df["Best Possible Score"]
    )
    results_df["Normalized Regret (%)"] = 100 * results_df["Normalized Regret"]

    best_possible_by_profile = (
        results_df.drop_duplicates("Profile ID")
        .set_index("Profile ID")["Best Possible Score"]
    )
    _add_incumbent_regret(convergence_df, best_possible_by_profile)

    return results_df, convergence_df


normalized_family_results = []
normalized_family_convergence = []
for sensitivity_spec in sensitivity_family_specs:
    family_results, family_convergence = _normalize_sensitivity_family(
        sensitivity_spec["family_name"],
        *_get_sensitivity_sources(sensitivity_spec),
    )
    normalized_family_results.append(family_results)
    normalized_family_convergence.append(family_convergence)

sensitivity_results_df = pd.concat(normalized_family_results, ignore_index=True)
sensitivity_convergence_df = pd.concat(
    normalized_family_convergence,
    ignore_index=True,
)
sensitivity_results_df["Profile Rank"] = sensitivity_results_df.groupby(
    ["Profile ID", "Iteration Budget"]
)["Normalized Regret"].rank(method="average", ascending=True)


def _bootstrap_mean_ci(values, seed):
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        mean_value = float(np.mean(values))
        return mean_value, mean_value
    rng = np.random.default_rng(seed)
    bootstrap_means = rng.choice(
        values,
        size=(sensitivity_n_bootstrap, len(values)),
        replace=True,
    ).mean(axis=1)
    return tuple(np.quantile(bootstrap_means, [0.025, 0.975]))


def _paired_sign_flip(values_a, values_b, seed):
    differences = np.asarray(values_a, dtype=float) - np.asarray(values_b, dtype=float)
    observed_difference = float(np.mean(differences))
    rng = np.random.default_rng(seed)
    signs = rng.choice(
        np.array([-1.0, 1.0]),
        size=(sensitivity_n_permutations, len(differences)),
    )
    permuted_means = np.mean(signs * differences, axis=1)
    p_value = (
        1
        + np.count_nonzero(np.abs(permuted_means) >= abs(observed_difference))
    ) / (sensitivity_n_permutations + 1)
    return differences, observed_difference, p_value


def _paired_range_permutation(wide, seed):
    values = wide.to_numpy(dtype=float)
    observed_statistic = float(np.ptp(values.mean(axis=0)))
    rng = np.random.default_rng(seed)
    permuted_statistics = np.empty(sensitivity_n_permutations)
    for permutation_index in range(sensitivity_n_permutations):
        permuted_values = np.array(
            [rng.permutation(profile_values) for profile_values in values]
        )
        permuted_statistics[permutation_index] = np.ptp(
            permuted_values.mean(axis=0)
        )
    p_value = (
        1
        + np.count_nonzero(permuted_statistics >= observed_statistic)
    ) / (sensitivity_n_permutations + 1)
    return observed_statistic, p_value


def _holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan)
    valid = np.isfinite(p_values)
    if valid.any():
        adjusted[valid] = multipletests(p_values[valid], method="holm")[1]
    return adjusted


family_method_summary_rows = []
for group_index, (group_key, group) in enumerate(
    sensitivity_results_df.groupby(
        ["Iteration Budget", "Surface Family", "Method"],
        sort=True,
    )
):
    iteration_budget, family_name, method = group_key
    regret_values = group["Normalized Regret (%)"].to_numpy(dtype=float)
    ci_low, ci_high = _bootstrap_mean_ci(
        regret_values,
        sensitivity_random_seed + group_index,
    )
    family_method_summary_rows.append(
        {
            "Iteration Budget": iteration_budget,
            "Surface Family": family_name,
            "Method": method,
            "N Profiles": len(group),
            "Mean Normalized Regret (%)": np.mean(regret_values),
            "95% CI Low (%)": ci_low,
            "95% CI High (%)": ci_high,
            "Median Normalized Regret (%)": np.median(regret_values),
            "Q25 Normalized Regret (%)": np.quantile(regret_values, 0.25),
            "Q75 Normalized Regret (%)": np.quantile(regret_values, 0.75),
            "P Regret <= 5%": np.mean(
                group["Normalized Regret"] <= sensitivity_regret_tolerance
            ),
            "Mean Profile Rank": group["Profile Rank"].mean(),
            "P Best Method": np.mean(group["Profile Rank"] == 1),
            "Mean Seconds": (
                group["Sec To Compute"].mean()
                if "Sec To Compute" in group.columns
                else np.nan
            ),
        }
    )

sensitivity_family_method_df = pd.DataFrame(family_method_summary_rows)
sensitivity_family_method_df["Family Mean Regret Rank"] = (
    sensitivity_family_method_df.groupby(
        ["Iteration Budget", "Surface Family"]
    )["Mean Normalized Regret (%)"].rank(method="average", ascending=True)
)
sensitivity_family_method_matrix_df = (
    sensitivity_family_method_df.pivot(
        index=["Iteration Budget", "Surface Family"],
        columns="Method",
        values="Mean Normalized Regret (%)",
    )
    .reindex(columns=sensitivity_methods)
    .reset_index()
    .rename_axis(None, axis=1)
)

family_winner_rows = []
for group_key, group in sensitivity_family_method_df.groupby(
    ["Iteration Budget", "Surface Family"],
    sort=True,
):
    iteration_budget, family_name = group_key
    best_value = group["Mean Normalized Regret (%)"].min()
    winners = group[np.isclose(group["Mean Normalized Regret (%)"], best_value)][
        "Method"
    ].tolist()
    family_winner_rows.append(
        {
            "Iteration Budget": iteration_budget,
            "Surface Family": family_name,
            "Best Method(s)": "; ".join(winners),
            "Best Mean Normalized Regret (%)": best_value,
            "N Methods Tied": len(winners),
        }
    )
sensitivity_family_winners_df = pd.DataFrame(family_winner_rows)

overall_summary_rows = []
for budget_index, (iteration_budget, budget_frame) in enumerate(
    sensitivity_results_df.groupby("Iteration Budget", sort=True)
):
    budget_winner_rows = sensitivity_family_winners_df[
        sensitivity_family_winners_df["Iteration Budget"] == iteration_budget
    ]
    winner_counts = {
        method: int(
            budget_winner_rows["Best Method(s)"].map(
                lambda winner_text: method in {
                    winner.strip()
                    for winner in winner_text.split(";")
                }
            ).sum()
        )
        for method in sensitivity_methods
    }
    for method_index, method in enumerate(sensitivity_methods):
        method_frame = budget_frame[budget_frame["Method"] == method]
        profile_regrets = method_frame["Normalized Regret (%)"].to_numpy(dtype=float)
        profile_ci_low, profile_ci_high = _bootstrap_mean_ci(
            profile_regrets,
            sensitivity_random_seed + 1000 + budget_index * 100 + method_index,
        )
        overall_summary_rows.append(
            {
                "Method": method,
                "N Profiles": len(profile_regrets),
                "Mean Regret (%)": np.mean(profile_regrets),
                "95% CI Low (%)": profile_ci_low,
                "95% CI High (%)": profile_ci_high,
                "Median Regret (%)": np.median(profile_regrets),
                "Success Rate (Regret <= 5%)": np.mean(
                    method_frame["Normalized Regret"] <= sensitivity_regret_tolerance
                ),
                "Mean Rank (1-4)": method_frame["Profile Rank"].mean(),
                "Families Won (of 9)": winner_counts[method],
                "Mean Seconds": (
                    method_frame["Sec To Compute"].mean()
                    if "Sec To Compute" in method_frame.columns
                    else np.nan
                ),
            }
        )
sensitivity_overall_summary_df = pd.DataFrame(overall_summary_rows)

sensitivity_global_omnibus_rows = []
sensitivity_global_pairwise_rows = []
sensitivity_family_interaction_rows = []
for budget_index, (iteration_budget, budget_frame) in enumerate(
    sensitivity_results_df.groupby("Iteration Budget", sort=True)
):
    wide = (
        budget_frame.pivot(
            index="Profile ID",
            columns="Method",
            values="Normalized Regret",
        )
        .reindex(columns=sensitivity_methods)
        .dropna()
    )
    if len(wide) < 2:
        continue
    friedman_statistic, friedman_p_value = stats.friedmanchisquare(
        *[wide[method].to_numpy(dtype=float) for method in sensitivity_methods]
    )
    permutation_statistic, permutation_p_value = _paired_range_permutation(
        wide,
        sensitivity_random_seed + 3000 + budget_index,
    )
    sensitivity_global_omnibus_rows.append(
        {
            "Iteration Budget": iteration_budget,
            "N Paired Profiles": len(wide),
            "N Families": budget_frame["Surface Family"].nunique(),
            "Endpoint": "Selected Score (Noise-Free), normalized regret",
            "Friedman Statistic": friedman_statistic,
            "Friedman P Value": friedman_p_value,
            "Kendall's W": friedman_statistic
            / (len(wide) * (len(sensitivity_methods) - 1)),
            "Paired Permutation Range Statistic": permutation_statistic,
            "Paired Permutation P Value": permutation_p_value,
        }
    )

    for pair_index, (method_a, method_b) in enumerate(
        itertools.combinations(sensitivity_methods, 2)
    ):
        differences, mean_difference, p_value = _paired_sign_flip(
            wide[method_a],
            wide[method_b],
            sensitivity_random_seed
            + 4000
            + budget_index * 100
            + pair_index,
        )
        ci_low, ci_high = _bootstrap_mean_ci(
            100 * differences,
            sensitivity_random_seed
            + 5000
            + budget_index * 100
            + pair_index,
        )
        difference_sd = np.std(differences, ddof=1)
        sensitivity_global_pairwise_rows.append(
            {
                "Iteration Budget": iteration_budget,
                "Method A": method_a,
                "Method B": method_b,
                "N Paired Profiles": len(differences),
                "Mean Regret Difference (A - B) (%)": 100 * mean_difference,
                "95% CI Low (%)": ci_low,
                "95% CI High (%)": ci_high,
                "Paired Effect Size dz": (
                    mean_difference / difference_sd
                    if difference_sd > 0
                    else np.nan
                ),
                "P Method A Lower Regret": np.mean(differences < 0),
                "Paired Sign-Flip P Value": p_value,
            }
        )

    for pair_index, (method_a, method_b) in enumerate(
        itertools.combinations(sensitivity_methods, 2)
    ):
        family_difference_arrays = []
        family_mean_differences = []
        for _, family_frame in budget_frame.groupby("Surface Family"):
            family_wide = (
                family_frame.pivot(
                    index="Profile ID",
                    columns="Method",
                    values="Normalized Regret",
                )
                .reindex(columns=sensitivity_methods)
                .dropna()
            )
            if len(family_wide) > 0:
                family_differences = (
                    family_wide[method_a].to_numpy(dtype=float)
                    - family_wide[method_b].to_numpy(dtype=float)
                )
                family_difference_arrays.append(family_differences)
                family_mean_differences.append(np.mean(family_differences))
        if len(family_difference_arrays) >= 2:
            try:
                heterogeneity_statistic, heterogeneity_p_value = stats.kruskal(
                    *family_difference_arrays
                )
            except ValueError:
                heterogeneity_statistic, heterogeneity_p_value = np.nan, np.nan
        else:
            heterogeneity_statistic, heterogeneity_p_value = np.nan, np.nan
        sensitivity_family_interaction_rows.append(
            {
                "Iteration Budget": iteration_budget,
                "Method A": method_a,
                "Method B": method_b,
                "N Families": len(family_difference_arrays),
                "Mean Family Contrast (A - B) (%)": 100
                * np.mean(family_mean_differences),
                "SD Family Contrast (%)": (
                    100 * np.std(family_mean_differences, ddof=1)
                    if len(family_mean_differences) > 1
                    else np.nan
                ),
                "Min Family Contrast (%)": 100 * np.min(family_mean_differences),
                "Max Family Contrast (%)": 100 * np.max(family_mean_differences),
                "Kruskal-Wallis H": heterogeneity_statistic,
                "Family Contrast Heterogeneity P Value": heterogeneity_p_value,
            }
        )

sensitivity_global_omnibus_df = pd.DataFrame(sensitivity_global_omnibus_rows)
sensitivity_global_pairwise_df = pd.DataFrame(sensitivity_global_pairwise_rows)
if not sensitivity_global_pairwise_df.empty:
    sensitivity_global_pairwise_df["Holm Adjusted P Value"] = np.nan
    for iteration_budget in sensitivity_global_pairwise_df["Iteration Budget"].unique():
        budget_mask = (
            sensitivity_global_pairwise_df["Iteration Budget"] == iteration_budget
        )
        sensitivity_global_pairwise_df.loc[budget_mask, "Holm Adjusted P Value"] = _holm_adjust(
            sensitivity_global_pairwise_df.loc[
                budget_mask,
                "Paired Sign-Flip P Value",
            ]
        )
sensitivity_family_interaction_df = pd.DataFrame(
    sensitivity_family_interaction_rows
)
if not sensitivity_family_interaction_df.empty:
    sensitivity_family_interaction_df["Holm Adjusted P Value"] = np.nan
    for iteration_budget in sensitivity_family_interaction_df["Iteration Budget"].unique():
        budget_mask = sensitivity_family_interaction_df["Iteration Budget"] == iteration_budget
        sensitivity_family_interaction_df.loc[
            budget_mask,
            "Holm Adjusted P Value",
        ] = _holm_adjust(
            sensitivity_family_interaction_df.loc[
                budget_mask,
                "Family Contrast Heterogeneity P Value",
            ]
        )

sensitivity_convergence_summary_df = pd.DataFrame()
sensitivity_convergence_compact_df = pd.DataFrame()
trajectory_score_column = INCUMBENT_REGRET_COLUMN
if trajectory_score_column in sensitivity_convergence_df.columns and "Query" in sensitivity_convergence_df.columns:
    trajectory_group_columns = [
        "Surface Family",
        "Method",
        "Iteration Budget",
        "Query",
    ]
    sensitivity_convergence_summary_df = (
        sensitivity_convergence_df.groupby(trajectory_group_columns)[
            trajectory_score_column
        ]
        .agg(
            Median="median",
            Q25=lambda values: values.quantile(0.25),
            Q75=lambda values: values.quantile(0.75),
        )
        .reset_index()
    )
    profile_counts = (
        sensitivity_convergence_df.groupby(trajectory_group_columns)["Profile ID"]
        .nunique()
        .reset_index(name="N Profiles")
    )
    sensitivity_convergence_summary_df = sensitivity_convergence_summary_df.merge(
        profile_counts,
        on=trajectory_group_columns,
        how="left",
    )
    expected_profiles_by_family = sensitivity_convergence_df.groupby(
        "Surface Family"
    )["Profile ID"].nunique()
    sensitivity_convergence_summary_df["N Profiles Expected"] = (
        sensitivity_convergence_summary_df["Surface Family"].map(
            expected_profiles_by_family
        )
    )
    sensitivity_convergence_summary_df["Complete Coverage"] = (
        sensitivity_convergence_summary_df["N Profiles"]
        == sensitivity_convergence_summary_df["N Profiles Expected"]
    )
    compact_rows = []
    compact_group_columns = [
        "Surface Family",
        "Method",
        "Iteration Budget",
    ]
    for group_key, group in sensitivity_convergence_summary_df.groupby(
        compact_group_columns,
        sort=True,
    ):
        ordered_group = group.sort_values("Query")
        ordered_group = ordered_group.dropna(subset=["Median"])
        if ordered_group.empty:
            continue
        first_row = ordered_group.iloc[0]
        final_row = ordered_group.iloc[-1]
        compact_rows.append(
            {
                **dict(zip(compact_group_columns, group_key)),
                "N Profiles": int(final_row["N Profiles"]),
                "N Queries": len(ordered_group),
                "All Queries Fully Covered": bool(
                    ordered_group["Complete Coverage"].all()
                ),
                "First Query": first_row["Query"],
                "Final Query": final_row["Query"],
                "First Median Normalized Regret (%)": first_row["Median"],
                "Final Median Normalized Regret (%)": final_row["Median"],
                "Final Q25 (%)": final_row["Q25"],
                "Final Q75 (%)": final_row["Q75"],
                "Median Regret Improvement (%)": (
                    first_row["Median"] - final_row["Median"]
                ),
            }
        )
    sensitivity_convergence_compact_df = pd.DataFrame(compact_rows)

sensitivity_crn_audit_df = pd.DataFrame()
sensitivity_crn_summary_df = pd.DataFrame()
crn_columns = {
    "Profile ID",
    "Iteration Budget",
    "Query",
    "Method",
    "Evaluation Seed",
}
if crn_columns.issubset(sensitivity_convergence_df.columns):
    sensitivity_crn_audit_df = (
        sensitivity_convergence_df.groupby(
            ["Surface Family", "Profile ID", "Iteration Budget", "Query"]
        )
        .agg(
            N_Methods=("Method", "nunique"),
            N_Evaluation_Seeds=("Evaluation Seed", "nunique"),
        )
        .reset_index()
    )
    sensitivity_crn_audit_df["Uses Common Random Number"] = (
        (sensitivity_crn_audit_df["N_Methods"] == len(sensitivity_methods))
        & (sensitivity_crn_audit_df["N_Evaluation_Seeds"] == 1)
    )
    sensitivity_crn_summary_df = (
        sensitivity_crn_audit_df.groupby(["Surface Family", "Iteration Budget"])[
            "Uses Common Random Number"
        ]
        .agg(
            **{
                "N Tested Groups": "size",
                "N Passed Groups": "sum",
                "N Failed Groups": lambda values: (~values).sum(),
            }
        )
        .reset_index()
    )
    sensitivity_crn_summary_df["Overall Status"] = np.where(
        sensitivity_crn_summary_df["N Failed Groups"] == 0,
        "PASS",
        "FAIL",
    )

sensitivity_validation_rows = []
for family_name, family_frame in sensitivity_results_df.groupby(
    "Surface Family",
    sort=True,
):
    family_budgets = sorted(family_frame["Iteration Budget"].unique().tolist())
    sensitivity_validation_rows.append(
        {
            "Surface Family": family_name,
            "N Profiles": family_frame["Profile ID"].nunique(),
            "Iteration Budgets": ", ".join(str(value) for value in family_budgets),
            "N Methods": family_frame["Method"].nunique(),
            "Complete Paired Profiles": (
                family_frame.groupby(["Profile ID", "Iteration Budget"])["Method"]
                .nunique()
                .eq(len(sensitivity_methods))
                .all()
            ),
            "CRN Audit Status": (
                ", ".join(
                    sensitivity_crn_summary_df.loc[
                        sensitivity_crn_summary_df["Surface Family"] == family_name,
                        "Overall Status",
                    ].astype(str)
                )
                if not sensitivity_crn_summary_df.empty
                else "NOT AVAILABLE"
            ),
        }
    )
sensitivity_validation_df = pd.DataFrame(sensitivity_validation_rows)

sensitivity_export_frames = {
    "combined_sensitivity_profile_results.csv": sensitivity_results_df,
    "combined_sensitivity_convergence.csv": sensitivity_convergence_df,
    "sensitivity_family_method_summary.csv": sensitivity_family_method_df,
    "sensitivity_family_method_matrix.csv": sensitivity_family_method_matrix_df,
    "sensitivity_family_winners.csv": sensitivity_family_winners_df,
    "sensitivity_overall_summary.csv": sensitivity_overall_summary_df,
    "sensitivity_global_omnibus.csv": sensitivity_global_omnibus_df,
    "sensitivity_global_pairwise.csv": sensitivity_global_pairwise_df,
    "sensitivity_family_interaction.csv": sensitivity_family_interaction_df,
    "sensitivity_convergence_summary.csv": sensitivity_convergence_summary_df,
    "sensitivity_convergence_compact.csv": sensitivity_convergence_compact_df,
    "sensitivity_crn_audit.csv": sensitivity_crn_audit_df,
    "sensitivity_crn_summary.csv": sensitivity_crn_summary_df,
}
sensitivity_output_files = {}
for filename, frame in sensitivity_export_frames.items():
    output_path = Path(filename)
    frame.to_csv(output_path, index=False)
    sensitivity_output_files[filename] = str(output_path)

print(
    "Primary endpoint: Selected Score (Noise-Free), expressed as normalized "
    "regret; lower regret is better."
)
print(
    "Convergence endpoint: noise-free regret of the incumbent, i.e. the "
    "parameters the optimizer would recommend if stopped at that query."
)
if sensitivity_results_df["Iteration Budget"].nunique() != 1:
    print(
        "The formal families use different iteration budgets. Results are "
        "reported separately by budget; rerun all formal families at one common "
        "budget before reporting a single pooled conclusion."
    )

print("\n--- Sensitivity input validation ---")
display(sensitivity_validation_df)
print("\n--- Family-by-method effectiveness table ---")
display(sensitivity_family_method_df)
print("\n--- Mean normalized regret matrix (%) ---")
display(sensitivity_family_method_matrix_df)
print("\n--- Best method by family ---")
display(sensitivity_family_winners_df)
print("\n--- Overall optimizer performance (all 360 profiles) ---")
display(sensitivity_overall_summary_df)
print("\n--- Global paired omnibus tests, stratified by iteration budget ---")
display(sensitivity_global_omnibus_df)
print("\n--- Global paired post-hoc tests with Holm correction ---")
display(sensitivity_global_pairwise_df)
print("\n--- Family-by-method contrast heterogeneity ---")
display(sensitivity_family_interaction_df)
if not sensitivity_convergence_compact_df.empty:
    print("\n--- Compact cross-family convergence summary ---")
    display(sensitivity_convergence_compact_df)
if not sensitivity_crn_summary_df.empty:
    print("\n--- Compact cross-family common-random-number audit ---")
    display(sensitivity_crn_summary_df)
print(f"\nExported {len(sensitivity_output_files)} sensitivity-analysis CSV files.")

sensitivity_analysis_outputs = {
    "profile_results_df": sensitivity_results_df,
    "convergence_df": sensitivity_convergence_df,
    "family_method_summary_df": sensitivity_family_method_df,
    "family_method_matrix_df": sensitivity_family_method_matrix_df,
    "family_winners_df": sensitivity_family_winners_df,
    "overall_summary_df": sensitivity_overall_summary_df,
    "global_omnibus_df": sensitivity_global_omnibus_df,
    "global_pairwise_df": sensitivity_global_pairwise_df,
    "family_interaction_df": sensitivity_family_interaction_df,
    "convergence_summary_df": sensitivity_convergence_summary_df,
    "convergence_compact_df": sensitivity_convergence_compact_df,
    "crn_audit_df": sensitivity_crn_audit_df,
    "crn_summary_df": sensitivity_crn_summary_df,
    "validation_df": sensitivity_validation_df,
    "trajectory_score_column": trajectory_score_column,
    "files": sensitivity_output_files,
}

checkpoint_queries = [20, 40]
checkpoint_columns = [
    "Iteration Budget",
    "Surface Family",
    "Query",
    "N Profiles",
    "BOGP",
    "TPE",
    "SMAC",
    "Random Search",
]

if sensitivity_convergence_summary_df.empty:
    sensitivity_convergence_checkpoint_df = pd.DataFrame(
        columns=checkpoint_columns
    )
else:
    checkpoint_rows = sensitivity_convergence_summary_df.loc[
        sensitivity_convergence_summary_df["Query"].isin(checkpoint_queries)
        & sensitivity_convergence_summary_df["Complete Coverage"]
    ].copy()

    if checkpoint_rows.empty:
        sensitivity_convergence_checkpoint_df = pd.DataFrame(
            columns=checkpoint_columns
        )
    else:
        checkpoint_rows["Median [Q1, Q3] (%)"] = checkpoint_rows.apply(
            lambda row: (
                f"{row['Median']:.1f} "
                f"[{row['Q25']:.1f}, {row['Q75']:.1f}]"
            ),
            axis=1,
        )
        sensitivity_convergence_checkpoint_df = (
            checkpoint_rows.pivot(
                index=[
                    "Iteration Budget",
                    "Surface Family",
                    "Query",
                    "N Profiles",
                ],
                columns="Method",
                values="Median [Q1, Q3] (%)",
            )
            .reindex(columns=sensitivity_methods)
            .rename(
                columns={
                    "Bayesian Optimization": "BOGP",
                    "TPE Optimization": "TPE",
                    "SMAC Optimization": "SMAC",
                }
            )
            .reset_index()
            .reindex(columns=checkpoint_columns)
            .sort_values(
                ["Iteration Budget", "Surface Family", "Query"]
            )
            .reset_index(drop=True)
        )

print(
    f"Checkpoint metric: {trajectory_score_column}; values are median [Q1, Q3] "
    "normalized regret (%). Only fully covered family-method-query groups are included."
)
display(sensitivity_convergence_checkpoint_df)

checkpoint_output_path = Path("sensitivity_convergence_checkpoints.csv")
sensitivity_convergence_checkpoint_df.to_csv(checkpoint_output_path, index=False)
sensitivity_output_files[checkpoint_output_path.name] = str(checkpoint_output_path)
sensitivity_analysis_outputs["convergence_checkpoints_df"] = (
    sensitivity_convergence_checkpoint_df
)
sensitivity_analysis_outputs["files"] = sensitivity_output_files
print(f"Exported checkpoint table to {checkpoint_output_path.resolve()}")


In [ ]:
seed_block_required_inputs = [
    "sensitivity_results_df",
    "sensitivity_methods",
    "sensitivity_n_permutations",
    "sensitivity_n_bootstrap",
    "sensitivity_random_seed",
    "sensitivity_regret_tolerance",
    "_paired_range_permutation",
    "_paired_sign_flip",
    "_bootstrap_mean_ci",
    "_holm_adjust",
]
missing_seed_block_inputs = [
    name for name in seed_block_required_inputs if name not in globals()
]
if missing_seed_block_inputs:
    raise RuntimeError(
        "Run the preceding sensitivity-analysis cell after loading all nine family "
        "result sets. Missing seed-block inputs: "
        + ", ".join(missing_seed_block_inputs)
    )

print("SEED-BLOCK CORRECTION: The row-wise pooled omnibus/pairwise tests and pooled CIs printed above treat repeated family seeds as independent. They are superseded by the seed-block results below.")


def _paired_friedman_with_complete_tie_fallback(wide):
    values = wide.to_numpy(dtype=float)
    if np.allclose(values, values[:, :1], rtol=1e-12, atol=1e-12):
        return 0.0, 1.0
    result = stats.friedmanchisquare(
        *[wide[column].to_numpy(dtype=float) for column in wide.columns]
    )
    return float(result.statistic), float(result.pvalue)


seed_block_results = sensitivity_results_df.copy()
seed_block_seed = pd.to_numeric(seed_block_results["Seed"], errors="coerce")
if seed_block_seed.isna().any() or not np.equal(seed_block_seed, np.floor(seed_block_seed)).all():
    raise ValueError("Seed-block analysis requires integer-convertible Seed values.")
seed_block_results["Seed"] = seed_block_seed.astype("int64")

seed_block_duplicate_rows = seed_block_results.duplicated(
    subset=["Iteration Budget", "Surface Family", "Seed", "Method"],
    keep=False,
)
if seed_block_duplicate_rows.any():
    raise ValueError("Seed-block analysis found duplicate family-seed-method rows.")

seed_block_pairwise_rows = []
seed_block_omnibus_rows = []
seed_block_overall_rows = []
seed_block_family_contrast_rows = []
seed_block_result_frames = []

for budget_index, (iteration_budget, budget_data) in enumerate(
    seed_block_results.groupby("Iteration Budget", sort=True)
):
    family_names = sorted(budget_data["Surface Family"].astype(str).unique())
    n_families = len(family_names)
    if n_families < 1:
        continue

    family_profile_wide = budget_data.pivot(
        index=["Seed", "Surface Family"],
        columns="Method",
        values="Normalized Regret",
    ).reindex(columns=sensitivity_methods)

    complete_family_profiles = family_profile_wide.dropna()
    family_counts_by_seed = complete_family_profiles.groupby(level="Seed").size()
    complete_seed_ids = family_counts_by_seed[
        family_counts_by_seed == n_families
    ].index
    complete_family_profiles = complete_family_profiles.loc[
        complete_family_profiles.index.get_level_values("Seed").isin(complete_seed_ids)
    ]
    seed_block_wide = complete_family_profiles.groupby(level="Seed", sort=True).mean()
    seed_block_wide = seed_block_wide.reindex(columns=sensitivity_methods)

    if len(seed_block_wide) < 2:
        raise ValueError(
            f"Budget {iteration_budget} has fewer than two complete seed blocks "
            f"across its {n_families} families."
        )

    n_seed_blocks = len(seed_block_wide)
    n_candidate_seeds = budget_data["Seed"].nunique()
    n_excluded_seed_blocks = n_candidate_seeds - n_seed_blocks
    complete_budget_data = budget_data[
        budget_data["Seed"].isin(complete_seed_ids)
    ].copy()

    seed_block_long = seed_block_wide.rename_axis("Seed").reset_index()
    seed_block_long.insert(0, "N Families", n_families)
    seed_block_long.insert(0, "Iteration Budget", iteration_budget)
    seed_block_result_frames.append(seed_block_long)

    friedman_statistic, friedman_p_value = _paired_friedman_with_complete_tie_fallback(
        seed_block_wide
    )
    permutation_statistic, permutation_p_value = _paired_range_permutation(
        seed_block_wide,
        sensitivity_random_seed + 3000 + budget_index,
    )
    seed_block_omnibus_rows.append(
        {
            "Iteration Budget": iteration_budget,
            "N Paired Seed Blocks": n_seed_blocks,
            "N Families": n_families,
            "N Excluded Seed Blocks": n_excluded_seed_blocks,
            "Family Weighting": "Equal",
            "Endpoint": "Mean normalized noise-free regret across families",
            "Friedman Statistic": friedman_statistic,
            "Friedman P Value": friedman_p_value,
            "Kendall's W": friedman_statistic
            / (n_seed_blocks * (len(sensitivity_methods) - 1)),
            "Paired Permutation Range Statistic": permutation_statistic,
            "Paired Permutation P Value": permutation_p_value,
        }
    )

    method_family_means = complete_budget_data.groupby(
        ["Surface Family", "Method"]
    )["Normalized Regret (%)"].mean()
    winner_counts = {method: 0 for method in sensitivity_methods}
    for family_name in family_names:
        family_means = method_family_means.loc[family_name]
        best_mean = family_means.min()
        for method in sensitivity_methods:
            if np.isclose(family_means[method], best_mean):
                winner_counts[method] += 1

    for method_index, method in enumerate(sensitivity_methods):
        block_regrets = 100 * seed_block_wide[method].to_numpy(dtype=float)
        ci_low, ci_high = _bootstrap_mean_ci(
            block_regrets,
            sensitivity_random_seed + 1000 + budget_index * 100 + method_index,
        )
        method_data = complete_budget_data[complete_budget_data["Method"] == method]
        seed_block_overall_rows.append(
            {
                "Iteration Budget": iteration_budget,
                "Method": method,
                "N Profiles": len(method_data),
                "N Seed Blocks": n_seed_blocks,
                "N Families": n_families,
                "Mean Regret (%)": np.mean(block_regrets),
                "95% Seed-Block CI Low (%)": ci_low,
                "95% Seed-Block CI High (%)": ci_high,
                "Median Family-Profile Regret (%)": method_data[
                    "Normalized Regret (%)"
                ].median(),
                "Success Rate (Regret <= 5%)": np.mean(
                    method_data["Normalized Regret"] <= sensitivity_regret_tolerance
                ),
                "Mean Profile Rank (descriptive)": method_data["Profile Rank"].mean(),
                "Families Won (of N Families)": winner_counts[method],
                "Mean Seconds": (
                    method_data["Sec To Compute"].mean()
                    if "Sec To Compute" in method_data.columns
                    else np.nan
                ),
            }
        )

    for pair_index, (method_a, method_b) in enumerate(
        itertools.combinations(sensitivity_methods, 2)
    ):
        differences, mean_difference, p_value = _paired_sign_flip(
            seed_block_wide[method_a],
            seed_block_wide[method_b],
            sensitivity_random_seed + 4000 + budget_index * 100 + pair_index,
        )
        ci_low, ci_high = _bootstrap_mean_ci(
            100 * differences,
            sensitivity_random_seed + 5000 + budget_index * 100 + pair_index,
        )
        difference_sd = np.std(differences, ddof=1)
        seed_block_pairwise_rows.append(
            {
                "Iteration Budget": iteration_budget,
                "Method A": method_a,
                "Method B": method_b,
                "N Paired Seed Blocks": n_seed_blocks,
                "N Families": n_families,
                "N Excluded Seed Blocks": n_excluded_seed_blocks,
                "Mean Regret Difference (A - B) (%)": 100 * mean_difference,
                "95% Seed-Block CI Low (%)": ci_low,
                "95% Seed-Block CI High (%)": ci_high,
                "Paired Effect Size dz": (
                    mean_difference / difference_sd if difference_sd > 0 else np.nan
                ),
                "Proportion Method A Lower Regret": np.mean(differences < 0),
                "Paired Sign-Flip P Value": p_value,
            }
        )

        family_contrast_wide = (
            complete_family_profiles[method_a] - complete_family_profiles[method_b]
        ).unstack(level="Surface Family").reindex(columns=family_names).dropna()
        if len(family_names) >= 3 and len(family_contrast_wide) >= 2:
            family_friedman_statistic, family_friedman_p_value = (
                _paired_friedman_with_complete_tie_fallback(family_contrast_wide)
            )
        else:
            family_friedman_statistic, family_friedman_p_value = np.nan, np.nan
        family_mean_contrasts = family_contrast_wide.mean(axis=0)
        seed_block_family_contrast_rows.append(
            {
                "Iteration Budget": iteration_budget,
                "Method A": method_a,
                "Method B": method_b,
                "N Paired Seed Blocks": len(family_contrast_wide),
                "N Families": len(family_names),
                "Mean Family Contrast (A - B) (%)": 100 * family_mean_contrasts.mean(),
                "SD Family Contrast (%)": 100 * family_mean_contrasts.std(ddof=1),
                "Min Family Contrast (%)": 100 * family_mean_contrasts.min(),
                "Max Family Contrast (%)": 100 * family_mean_contrasts.max(),
                "Repeated-Measures Friedman Statistic": family_friedman_statistic,
                "Family Contrast Heterogeneity P Value": family_friedman_p_value,
            }
        )

sensitivity_seed_block_results_df = pd.concat(seed_block_result_frames, ignore_index=True)
sensitivity_seed_block_overall_summary_df = pd.DataFrame(seed_block_overall_rows)
sensitivity_seed_block_omnibus_df = pd.DataFrame(seed_block_omnibus_rows)
sensitivity_seed_block_pairwise_df = pd.DataFrame(seed_block_pairwise_rows)
sensitivity_seed_block_family_contrast_df = pd.DataFrame(seed_block_family_contrast_rows)

for test_df, raw_column in [
    (sensitivity_seed_block_pairwise_df, "Paired Sign-Flip P Value"),
    (sensitivity_seed_block_family_contrast_df, "Family Contrast Heterogeneity P Value"),
]:
    if not test_df.empty:
        test_df["Holm Adjusted P Value"] = np.nan
        for iteration_budget in test_df["Iteration Budget"].unique():
            budget_mask = test_df["Iteration Budget"] == iteration_budget
            test_df.loc[budget_mask, "Holm Adjusted P Value"] = _holm_adjust(
                test_df.loc[budget_mask, raw_column]
            )

# Replace row-wise pooled inferential outputs so later notebook code uses the corrected results.
sensitivity_overall_summary_df = sensitivity_seed_block_overall_summary_df
sensitivity_global_omnibus_df = sensitivity_seed_block_omnibus_df
sensitivity_global_pairwise_df = sensitivity_seed_block_pairwise_df
sensitivity_family_interaction_df = sensitivity_seed_block_family_contrast_df

print("\n--- Seed-block overall optimizer performance ---")
print("Means and confidence intervals use equal family weighting and resample seed blocks.")
display(sensitivity_seed_block_overall_summary_df)
print("\n--- Seed-block paired omnibus tests by iteration budget ---")
display(sensitivity_seed_block_omnibus_df)
print("\n--- Seed-block paired optimizer comparisons with Holm correction ---")
display(sensitivity_seed_block_pairwise_df)
print("\n--- Family contrasts tested as repeated measures across seed blocks ---")
display(sensitivity_seed_block_family_contrast_df)

seed_block_export_frames = {
    "sensitivity_seed_block_results.csv": sensitivity_seed_block_results_df,
    "sensitivity_seed_block_overall_summary.csv": sensitivity_seed_block_overall_summary_df,
    "sensitivity_seed_block_omnibus.csv": sensitivity_seed_block_omnibus_df,
    "sensitivity_seed_block_pairwise.csv": sensitivity_seed_block_pairwise_df,
    "sensitivity_seed_block_family_contrasts.csv": sensitivity_seed_block_family_contrast_df,
    "sensitivity_overall_summary.csv": sensitivity_seed_block_overall_summary_df,
    "sensitivity_global_omnibus.csv": sensitivity_seed_block_omnibus_df,
    "sensitivity_global_pairwise.csv": sensitivity_seed_block_pairwise_df,
    "sensitivity_family_interaction.csv": sensitivity_seed_block_family_contrast_df,
}
for filename, frame in seed_block_export_frames.items():
    frame.to_csv(Path(filename), index=False)

if "sensitivity_output_files" in globals():
    sensitivity_output_files.update(
        {filename: str(Path(filename)) for filename in seed_block_export_frames}
    )
if "sensitivity_export_frames" in globals():
    sensitivity_export_frames.update(seed_block_export_frames)
if "sensitivity_analysis_outputs" in globals():
    sensitivity_analysis_outputs.update(
        {
            "overall_summary_df": sensitivity_seed_block_overall_summary_df,
            "global_omnibus_df": sensitivity_seed_block_omnibus_df,
            "global_pairwise_df": sensitivity_seed_block_pairwise_df,
            "family_interaction_df": sensitivity_seed_block_family_contrast_df,
            "seed_block_results_df": sensitivity_seed_block_results_df,
            "seed_block_overall_summary_df": sensitivity_seed_block_overall_summary_df,
            "seed_block_omnibus_df": sensitivity_seed_block_omnibus_df,
            "seed_block_pairwise_df": sensitivity_seed_block_pairwise_df,
            "seed_block_family_contrast_df": sensitivity_seed_block_family_contrast_df,
            "files": sensitivity_output_files,
        }
    )

print("\nSeed-block CSV outputs written. The earlier row-wise pooled inferential tables are superseded.")